# Mission 3 ver6 — 손실 함수와 증상별 token attention 비교

**기존 `mission3_ver5_inputs.zip` 그대로 사용합니다. 새 ZIP이나 ver5 결과는 필요 없습니다.**

1. Google Drive **공유 문서함**의 `mission3_ver5_inputs.zip`을 우클릭합니다.
2. **정리 → 바로가기 추가 → 내 드라이브**를 선택합니다. 내 드라이브 바로 아래에 추가하세요.
3. 이 노트북을 Colab에서 열고 GPU 런타임을 선택한 뒤 **모두 실행**합니다.
4. Drive 연결은 바로가기를 만든 계정으로 합니다. 다른 위치라면 첫 셀의 `INPUT_ZIP`만 수정하세요.

ZIP에는 기존 ver4 모델 5개, 보조 파일, Training 29,200개와 Validation 3,640개의 JSON이 있습니다.
공유 문서함의 별도 `DATA` 폴더를 연결하거나 WAV 30GB를 읽을 필요는 없습니다.
실행 코드가 이 노트북에 포함되어 있어 별도 `.py` 업로드도 필요 없습니다.

**`/content/mission3_ver5_inputs/DCC_AIdeas/results/mission3/ver4`는 압축을 푼 입력 모델입니다.**
**새 ver6 결과는 `내 드라이브/DCC_AIdeas/results/mission3/ver6`에 저장됩니다.**
ver4라고 표시된 입력 폴더를 ver6으로 이름을 바꾸지 마세요.

ver5에서는 추가 학습 뒤 점수가 낮아져 학습 전 가중치가 선택됐습니다.
ver6은 동일한 encoder 특징을 한 번 계산하여 재사용하고, 작은 분류기 실험 6개를 비교합니다.
A–C는 필요한 CLS·평균 요약을 메모리에 재사용하여 매 epoch마다 전체 token 특징을 읽는 비용도 줄입니다.
선택된 분류기 하나만 encoder 상위 2개 층과 최대 1 epoch 미세조정합니다.
기존 모델을 6번 복제하거나 encoder를 6번 처음부터 학습하지 않습니다.
단, 원래 Training fit·stop 26,320개 대화의 token 특징을 한 번 추출하고 저장하는 비용은 필요합니다.
별도 calibration 2,880개는 이 분류기 학습용 특징 캐시에서 제외합니다.

특징 캐시는 Colab 로컬 저장소에 두어 Drive 용량을 절약합니다. 같은 런타임에서는 재사용하며,
새 런타임에서는 ZIP을 다시 풉니다. 아직 완료하지 않은 분류기 실험이 있으면 특징도 다시 계산합니다.
분류기 비교가 끝나 미세조정만 재개하거나 완료 결과를 읽을 때는 특징 추출을 생략합니다.
실제 특징 캐시 필요 용량은 추출 전에 계산합니다.
기존 모델 5개도 결과 폴더에 중복 저장하지 않습니다. **입력 ZIP과 바로가기는 계속 보관하세요.**

목표는 Macro F1 개선이지만, **큰 성능 향상은 실행 전 보장할 수 없습니다.**
Training 내부에서 충분한 개선이 확인되지 않으면 기존 ver4 보정을 유지합니다.
공식 Validation은 최종 구성을 확정한 후 한 번 평가하며 모델 선택에 사용하지 않습니다.


## 실험 내용과 비교 원칙

| 실험 | 분류기와 학습 목표 | 확인할 점 |
|---|---|---|
| A | 기존 ver4 분류기 + Asymmetric Loss | 쉬운 음성 예제의 영향 완화 |
| B | 기존 ver4 분류기 + Distribution-Balanced Loss | Training fit의 빈도·동시 출현을 반영 |
| C | 기존 ver4 분류기 + ASL·soft Macro F1 | 학습 목표를 공식 F1에 더 가깝게 구성 |
| D | 직접적인 label-wise token attention + weighted BCE | 각 증상의 예측이 해당 token 표현에 직접 의존 |
| E | D + 분류기 R-Drop | 같은 특징을 두 dropout 경로로 예측해 일관성 학습 |
| F | E + 작은 label dependency 모듈 | fit 부분에서만 증상 간 관계 학습 |

D–F는 기존 logit에 0으로 시작하는 작은 출력을 더하는 ver5 구조와 다릅니다.
각 증상이 token을 직접 선택하고 window를 합쳐 최종 logit을 만듭니다.
동결 encoder 위의 R-Drop은 작은 분류기에서만 두 번 계산합니다.
최종 encoder 미세조정에는 기본적으로 두 번의 전체 순전파를 요구하는 R-Drop을 적용하지 않습니다.
soft Macro F1 보조항도 작은 분류기 단계에만 사용하며, 작은 encoder microbatch의 통계에는 적용하지 않습니다.

원래 ver4 fold 1의 **fit 21,032개**만 가중치·학습 통계 계산에 사용합니다.
**stop 5,288개**에서는 대화 중복 그룹을 분리한 3-fold threshold 교차평가 Macro F1으로
실험·epoch를 비교하고, 별도 **calibration 2,880개**에서는 5-fold 교차평가로 최종 혼합과 임계값을 비교합니다.
이는 threshold를 평가할 대화의 정답으로 직접 맞추지 않도록 한 비교입니다.
**5개 encoder를 각각 다시 학습해 얻는 5-fold 모델 OOF 결과는 아닙니다.**
같은 stop을 여러 후보에 사용하므로 이 점수를 새로운 데이터의 성능으로 해석하지 않습니다.

논문 원문을 검토한 적용 범위와 단순화는 포함된 `RESEARCH.md`에 기록합니다.
[PLM-ICD](https://aclanthology.org/2022.clinicalnlp-1.2/),
[LAAT](https://arxiv.org/abs/2007.06351),
[ASL](https://arxiv.org/abs/2009.14119),
[DB-Loss](https://arxiv.org/abs/2007.09654),
[SigmoidF1](https://arxiv.org/abs/2108.10566),
[R-Drop](https://proceedings.neurips.cc/paper_files/paper/2021/hash/5a66b9200f29ac3fa0ae244cc2a51b39-Abstract.html)의
아이디어를 한국어 9개 증상 문제에 맞추어 적용하며 논문의 결과를 그대로 재현한다고 주장하지 않습니다.
외부 데이터·의료 사전·LLM 생성 데이터는 추가하지 않습니다.
샘플의 label에 따라 입력 전처리를 바꾸지 않습니다. 손실의 정답·학습 통계 사용은 입력 전처리와 구분합니다.


In [ ]:
# 1. 공유 ZIP의 바로가기를 내 드라이브 바로 아래에 두었다면 그대로 실행하세요.
INPUT_ZIP = "/content/drive/MyDrive/mission3_ver5_inputs.zip"
BASE_DIR = None             # ZIP 대신 기존 ver4 폴더를 쓰려면 INPUT_ZIP=None과 함께 지정
TRAIN_LABEL_DIR = None      # ZIP에 포함된 Training JSON 자동 사용
VALID_LABEL_DIR = None      # ZIP에 포함된 Validation JSON 자동 사용
OUTPUT_DIR = None           # 기본: 내 드라이브/DCC_AIdeas/results/mission3/ver6
INFERENCE_CACHE_DIR = None  # 기본: 같은 results/mission3/prediction_cache (Drive)
FROZEN_FEATURE_CACHE_DIR = None  # 기본: /content/mission3_ver6_features (Colab 로컬)
RUN_VALIDATION = True       # 최종 모델을 확정한 뒤 Validation 전체를 평가

TRAIN_CONFIG = {
    "bundle_base": False,  # 기존 5개 모델 중복 저장 생략. 원래 ZIP을 보관하세요.
    "profiles": ["A", "B", "C", "D", "E", "F"],
    "head_epochs": 4,
    "head_batch_size": 32,
    "head_learning_rate": 3e-4,
    "head_patience": 2,
    "stop_crossfit_splits": 3,
    "finetune_epochs": 1,
    "finetune_top_layers": 2,
    "learning_rate": 2e-6,
    "finetune_head_learning_rate": 5e-5,
    "finetune_rdrop": False,
    "min_calibration_gain": 0.002,
    "min_calibration_improved_folds": 3,
}
INFERENCE_BATCH_SIZE = 8
# CUDA 메모리에 맞춰 encoder batch/window 크기를 자동 선택하고 유효 batch 32를 유지합니다.
# 설정을 바꾸어 새 실험을 만들 때는 OUTPUT_DIR을 새 폴더로 지정하세요.


In [ ]:
# 2. 기존 ver4 저장 모델과 호환되는 패키지 설치
import subprocess
import sys

REQUIREMENTS = ['torch>=2.6,<3', 'transformers==4.57.1', 'numpy==2.1.3', 'scikit-learn==1.6.1', 'scipy>=1.11,<2', 'iterative-stratification==0.1.9', 'joblib>=1.3,<2']
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *REQUIREMENTS], check=True)
print("환경 준비 완료. 학습과 추론은 새 Python 프로세스에서 실행합니다.")


In [ ]:
# 3. Drive 연결 → 기존 ZIP 검증·자동 압축 해제 → 입력과 새 결과 경로 확인
import json
import os
import tempfile
import time
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

preparation_started = time.monotonic()
START_DIR = Path.cwd().resolve()
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
roots = ([Path("/content"), Path("/content/drive/MyDrive")] if IN_COLAB else [])
roots += [START_DIR, *list(START_DIR.parents)[:6]]
roots = list(dict.fromkeys(roots))

BOOTSTRAP_SOURCE = '"""Restore the one-file Mission 3 input bundle using only the standard library.\n\nThe uploaded ZIP remains in Drive. Extraction uses a fixed local directory so\nepoch resume sees the same initial checkpoint path across Colab sessions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\nBUNDLE_NAME = "mission3_ver5_inputs.zip"\nMODEL_PREFIX = "DCC_AIdeas/results/mission3/ver4/"\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not name or "\\\\" in name or ":" in name or path.is_absolute()\n            or ".." in path.parts or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    path = root / _relative(name)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    return path\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef _files(manifest, archive):\n    if manifest.get("format") != "mission3_ver5_inputs_v1":\n        raise ValueError("Unsupported Mission 3 input bundle format")\n    files = manifest.get("files")\n    if not isinstance(files, list) or not files:\n        raise ValueError("Bundle manifest has no files")\n    names = set()\n    for row in files:\n        name = row["path"]\n        _relative(name)\n        if name in names or not name.startswith("DCC_AIdeas/"):\n            raise ValueError(f"Duplicate or unexpected bundle member: {name}")\n        names.add(name)\n        if (type(row["size"]) is not int or row["size"] < 0\n                or not re.fullmatch(r"[0-9a-f]{64}", row["sha256"])):\n            raise ValueError(f"Invalid manifest metadata: {name}")\n        info = archive.getinfo(name)\n        if info.is_dir() or info.file_size != row["size"]:\n            raise ValueError(f"Bundle size mismatch: {name}")\n    required = {MODEL_PREFIX + name for name in\n                ["mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz", "sparse.joblib",\n                 *[f"fold_{i}.ckpt" for i in range(1, 6)], "tokenizer/tokenizer.json",\n                 "tokenizer/tokenizer_config.json", "tokenizer/vocab.txt", "tokenizer/special_tokens_map.json"]}\n    if not required <= names:\n        raise ValueError(f"Missing ver4 inputs: {sorted(required - names)}")\n    for prefix, key in [(TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")]:\n        count = sum(name.startswith(prefix) and name.endswith(".json") for name in names)\n        if count < 1 or type(manifest.get(key)) is not int or count != manifest[key]:\n            raise ValueError(f"Bundle JSON count mismatch: {key}")\n    infos = archive.infolist()\n    if len({info.filename for info in infos}) != len(infos):\n        raise ValueError("Duplicate ZIP member")\n    allowed_files = names | {"bundle_manifest.json", "README_UPLOAD.txt"}\n    for info in infos:\n        _relative(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n        if not info.is_dir() and info.filename not in allowed_files:\n            raise ValueError(f"Unlisted bundle member: {info.filename}")\n    return files\n\n\ndef restore_input_bundle(archive_path, destination):\n    """Verify SHA256 while extracting; reuse unchanged files from a verified run."""\n    archive_path = Path(archive_path).expanduser().resolve()\n    root = Path(destination).expanduser().resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".mission3_bundle_complete.json"\n    with zipfile.ZipFile(archive_path) as archive:\n        manifest_bytes = archive.read("bundle_manifest.json")\n        manifest = json.loads(manifest_bytes)\n        files = _files(manifest, archive)\n        signature = hashlib.sha256(manifest_bytes).hexdigest()\n        previous = {}\n        try:\n            saved = json.loads(marker.read_text(encoding="utf-8"))\n            if saved.get("manifest_sha256") == signature:\n                previous = saved["files"]\n        except (OSError, ValueError, KeyError):\n            pass\n        pending, state = [], {}\n        for row in files:\n            target = _target(root, row["path"])\n            current = target.stat() if target.is_file() else None\n            stamp = [current.st_size, current.st_mtime_ns] if current else None\n            if stamp and current.st_size == row["size"] and previous.get(row["path"]) == stamp:\n                state[row["path"]] = stamp\n            else:\n                pending.append(row)\n        if not pending:\n            print("입력 ZIP: 이전에 검증한 Colab 로컬 파일을 재사용합니다.", flush=True)\n            return root / "DCC_AIdeas"\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Colab 로컬 저장소에 입력 ZIP을 풀 공간이 부족합니다.")\n        print(f"입력 ZIP 확인·압축 해제: {len(pending):,}개 파일 (WAV 없음)", flush=True)\n        for index, row in enumerate(pending, 1):\n            target = _target(root, row["path"])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Recover completed files from an interrupted extraction without copying them again.\n            verified = target.is_file() and target.stat().st_size == row["size"] and _hash(target) == row["sha256"]\n            if not verified:\n                temporary = None\n                try:\n                    digest, size = hashlib.sha256(), 0\n                    with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                            dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                        temporary = Path(writer.name)\n                        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n                            writer.write(chunk)\n                            digest.update(chunk)\n                            size += len(chunk)\n                    if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                        raise ValueError(f"ZIP 내용 검증 실패: {row[\'path\']}")\n                    temporary.replace(target)\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n            current = target.stat()\n            state[row["path"]] = [current.st_size, current.st_mtime_ns]\n            if row["size"] > 100 * 1024 * 1024 or index % 2000 == 0 or index == len(pending):\n                print(f"압축 해제 {index:,}/{len(pending):,}: {target.name}", flush=True)\n        # A marker is published only after every file has passed hash verification.\n        temporary = marker.with_suffix(".tmp")\n        temporary.write_text(json.dumps({"manifest_sha256": signature, "files": state}), encoding="utf-8")\n        temporary.replace(marker)\n    print(f"준비 완료: Training {manifest[\'training_count\']:,}개 / Validation {manifest[\'validation_count\']:,}개", flush=True)\n    return root / "DCC_AIdeas"\n'
bootstrap_namespace = {}
exec(compile(BOOTSTRAP_SOURCE, "mission3_bootstrap.py", "exec"), bootstrap_namespace)
bundle_path = Path(INPUT_ZIP).expanduser().resolve() if INPUT_ZIP is not None else None
if bundle_path is None and BASE_DIR is None:
    candidates = [root / relative for root in roots
                  for relative in ("mission3_ver5_inputs.zip", "DCC_AIdeas/mission3_ver5_inputs.zip",
                                   "outputs/upload/mission3_ver5_inputs.zip",
                                   "DCC_AIdeas/outputs/upload/mission3_ver5_inputs.zip")]
    bundle_path = next((path for path in candidates if path.is_file()), None)
BUNDLE_ROOT = None
if bundle_path is not None:
    if not bundle_path.is_file():
        raise FileNotFoundError(
            f"ZIP 바로가기를 찾지 못했습니다: {bundle_path}\n"
            "Drive 공유 문서함의 ZIP에서 '정리 → 바로가기 추가 → 내 드라이브'를 선택하세요. "
            "바로가기를 만든 계정으로 Drive를 연결하고 첫 셀의 INPUT_ZIP 경로를 확인하세요. "
            "바로가기 추가 전에 이미 연결했다면 drive.mount의 force_remount=True로 한 번 재연결하세요.")
    # 고정 경로를 유지해야 이전 실행의 체크포인트에서 재개할 수 있습니다.
    extract_dir = (Path("/content/mission3_ver5_inputs") if IN_COLAB
                   else Path(tempfile.gettempdir()) / "mission3_ver5_inputs")
    BUNDLE_ROOT = bootstrap_namespace["restore_input_bundle"](bundle_path, extract_dir)
    BASE_DIR = BUNDLE_ROOT / "results/mission3/ver4"

def locate(explicit, candidates, valid, name, required=True):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        if not valid(path):
            raise FileNotFoundError(f"{name} 경로를 확인하세요: {path}")
        return path
    for path in dict.fromkeys(candidates):
        if valid(path):
            return path.resolve()
    if required:
        raise FileNotFoundError(
            f"{name}을 찾지 못했습니다. 기존 mission3_ver5_inputs.zip의 바로가기를 내 드라이브에 추가하고 "
            "첫 셀의 INPUT_ZIP을 확인하세요.")
    return None

base_candidates = [path for root in roots for path in
                   (root, root / "results/mission3/ver4", root / "DCC_AIdeas/results/mission3/ver4")]
BASE_DIR = locate(BASE_DIR, base_candidates, lambda p: (p / "mission3_ver4_model.ckpt").is_file(), "BASE_DIR")
for name in ("splits.json", "calibration_predictions.npz", "sparse.joblib", "tokenizer",
             *[f"fold_{i}.ckpt" for i in range(1, 6)]):
    if not (BASE_DIR / name).exists():
        raise FileNotFoundError(f"기존 ver4 입력 파일이 빠졌습니다: {BASE_DIR / name}")

def label_candidates(name, split):
    values = ([BUNDLE_ROOT / "DATA/대학부 데이터" / split / "2.라벨링데이터" / name]
              if BUNDLE_ROOT is not None else [])
    for root in [*roots, *list(BASE_DIR.parents)[:4]]:
        values.append(root / name)
        for base in (root, root / "DATA", root / "DCC_AIdeas/DATA"):
            values.extend([base / split / "2.라벨링데이터" / name,
                           base / "대학부 데이터" / split / "2.라벨링데이터" / name])
    return values

has_json = lambda p: p.is_dir() and next(p.rglob("*.json"), None) is not None
TRAIN_LABEL_DIR = locate(TRAIN_LABEL_DIR, label_candidates("TL_서울_구급", "Training"), has_json, "TRAIN_LABEL_DIR")
VALID_LABEL_DIR = locate(VALID_LABEL_DIR, label_candidates("VL_서울_구급", "Validation"),
                         has_json, "VALID_LABEL_DIR", required=RUN_VALIDATION)
result_parent = Path("/content/drive/MyDrive/DCC_AIdeas/results/mission3") if IN_COLAB else BASE_DIR.parent
if not IN_COLAB and BUNDLE_ROOT is not None:
    result_parent = START_DIR / "results/mission3"
OUTPUT_DIR = (result_parent / "ver6" if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser()).resolve()
INFERENCE_CACHE_DIR = (result_parent / "prediction_cache" if INFERENCE_CACHE_DIR is None
                       else Path(INFERENCE_CACHE_DIR).expanduser()).resolve()
feature_default = Path("/content/mission3_ver6_features") if IN_COLAB else Path(tempfile.gettempdir()) / "mission3_ver6_features"
FROZEN_FEATURE_CACHE_DIR = (feature_default if FROZEN_FEATURE_CACHE_DIR is None
                            else Path(FROZEN_FEATURE_CACHE_DIR).expanduser()).resolve()
if OUTPUT_DIR == BASE_DIR or OUTPUT_DIR.is_relative_to(BASE_DIR) or BASE_DIR.is_relative_to(OUTPUT_DIR):
    raise ValueError("OUTPUT_DIR는 기존 ver4 입력 폴더와 분리된 새 폴더여야 합니다.")
TRAIN_CONFIG["feature_cache_dir"] = str(FROZEN_FEATURE_CACHE_DIR)
PREPARATION_MINUTES = (time.monotonic() - preparation_started) / 60
print("입력 ZIP:", bundle_path)
print("기존 ver4 입력 모델:", BASE_DIR)
print("Training JSON:", TRAIN_LABEL_DIR)
print("최종 평가 JSON:", VALID_LABEL_DIR)
print("새 ver6 결과 저장 위치:", OUTPUT_DIR)
print("동결 특징 캐시 (로컬):", FROZEN_FEATURE_CACHE_DIR)
print("공유 원시 예측 캐시 (Drive):", INFERENCE_CACHE_DIR)
print(f"Drive 연결·입력 준비 시간: {PREPARATION_MINUTES:.1f}분")


In [ ]:
# @title 4. 노트북에 포함된 ver6 실행 코드 준비 (별도 소스 업로드 불필요)
RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver6: warm-start token evidence model and calibrated ensemble."""\n', 'bootstrap.py': '"""Restore the one-file Mission 3 input bundle using only the standard library.\n\nThe uploaded ZIP remains in Drive. Extraction uses a fixed local directory so\nepoch resume sees the same initial checkpoint path across Colab sessions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\nBUNDLE_NAME = "mission3_ver5_inputs.zip"\nMODEL_PREFIX = "DCC_AIdeas/results/mission3/ver4/"\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not name or "\\\\" in name or ":" in name or path.is_absolute()\n            or ".." in path.parts or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    path = root / _relative(name)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    return path\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef _files(manifest, archive):\n    if manifest.get("format") != "mission3_ver5_inputs_v1":\n        raise ValueError("Unsupported Mission 3 input bundle format")\n    files = manifest.get("files")\n    if not isinstance(files, list) or not files:\n        raise ValueError("Bundle manifest has no files")\n    names = set()\n    for row in files:\n        name = row["path"]\n        _relative(name)\n        if name in names or not name.startswith("DCC_AIdeas/"):\n            raise ValueError(f"Duplicate or unexpected bundle member: {name}")\n        names.add(name)\n        if (type(row["size"]) is not int or row["size"] < 0\n                or not re.fullmatch(r"[0-9a-f]{64}", row["sha256"])):\n            raise ValueError(f"Invalid manifest metadata: {name}")\n        info = archive.getinfo(name)\n        if info.is_dir() or info.file_size != row["size"]:\n            raise ValueError(f"Bundle size mismatch: {name}")\n    required = {MODEL_PREFIX + name for name in\n                ["mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz", "sparse.joblib",\n                 *[f"fold_{i}.ckpt" for i in range(1, 6)], "tokenizer/tokenizer.json",\n                 "tokenizer/tokenizer_config.json", "tokenizer/vocab.txt", "tokenizer/special_tokens_map.json"]}\n    if not required <= names:\n        raise ValueError(f"Missing ver4 inputs: {sorted(required - names)}")\n    for prefix, key in [(TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")]:\n        count = sum(name.startswith(prefix) and name.endswith(".json") for name in names)\n        if count < 1 or type(manifest.get(key)) is not int or count != manifest[key]:\n            raise ValueError(f"Bundle JSON count mismatch: {key}")\n    infos = archive.infolist()\n    if len({info.filename for info in infos}) != len(infos):\n        raise ValueError("Duplicate ZIP member")\n    allowed_files = names | {"bundle_manifest.json", "README_UPLOAD.txt"}\n    for info in infos:\n        _relative(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n        if not info.is_dir() and info.filename not in allowed_files:\n            raise ValueError(f"Unlisted bundle member: {info.filename}")\n    return files\n\n\ndef restore_input_bundle(archive_path, destination):\n    """Verify SHA256 while extracting; reuse unchanged files from a verified run."""\n    archive_path = Path(archive_path).expanduser().resolve()\n    root = Path(destination).expanduser().resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".mission3_bundle_complete.json"\n    with zipfile.ZipFile(archive_path) as archive:\n        manifest_bytes = archive.read("bundle_manifest.json")\n        manifest = json.loads(manifest_bytes)\n        files = _files(manifest, archive)\n        signature = hashlib.sha256(manifest_bytes).hexdigest()\n        previous = {}\n        try:\n            saved = json.loads(marker.read_text(encoding="utf-8"))\n            if saved.get("manifest_sha256") == signature:\n                previous = saved["files"]\n        except (OSError, ValueError, KeyError):\n            pass\n        pending, state = [], {}\n        for row in files:\n            target = _target(root, row["path"])\n            current = target.stat() if target.is_file() else None\n            stamp = [current.st_size, current.st_mtime_ns] if current else None\n            if stamp and current.st_size == row["size"] and previous.get(row["path"]) == stamp:\n                state[row["path"]] = stamp\n            else:\n                pending.append(row)\n        if not pending:\n            print("입력 ZIP: 이전에 검증한 Colab 로컬 파일을 재사용합니다.", flush=True)\n            return root / "DCC_AIdeas"\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Colab 로컬 저장소에 입력 ZIP을 풀 공간이 부족합니다.")\n        print(f"입력 ZIP 확인·압축 해제: {len(pending):,}개 파일 (WAV 없음)", flush=True)\n        for index, row in enumerate(pending, 1):\n            target = _target(root, row["path"])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Recover completed files from an interrupted extraction without copying them again.\n            verified = target.is_file() and target.stat().st_size == row["size"] and _hash(target) == row["sha256"]\n            if not verified:\n                temporary = None\n                try:\n                    digest, size = hashlib.sha256(), 0\n                    with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                            dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                        temporary = Path(writer.name)\n                        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n                            writer.write(chunk)\n                            digest.update(chunk)\n                            size += len(chunk)\n                    if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                        raise ValueError(f"ZIP 내용 검증 실패: {row[\'path\']}")\n                    temporary.replace(target)\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n            current = target.stat()\n            state[row["path"]] = [current.st_size, current.st_mtime_ns]\n            if row["size"] > 100 * 1024 * 1024 or index % 2000 == 0 or index == len(pending):\n                print(f"압축 해제 {index:,}/{len(pending):,}: {target.name}", flush=True)\n        # A marker is published only after every file has passed hash verification.\n        temporary = marker.with_suffix(".tmp")\n        temporary.write_text(json.dumps({"manifest_sha256": signature, "files": state}), encoding="utf-8")\n        temporary.replace(marker)\n    print(f"준비 완료: Training {manifest[\'training_count\']:,}개 / Validation {manifest[\'validation_count\']:,}개", flush=True)\n    return root / "DCC_AIdeas"\n', 'neural.py': '"""Cached head ablations, direct label attention, and one bounded fine-tune.\n\nThe original fold\'s fit/stop identities are supplied by the pipeline. Six small\nheads share ONE frozen encoder feature pass; only the stop-selected winner is\nallowed an encoder fine-tune. Training calibration and official Validation\nlabels never enter this module.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn.utils.rnn import pad_sequence\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\nif __package__:\n    from .losses import MultilabelLoss, bernoulli_symmetric_kl\n    from .selection import StopSelector, better\n    from .ver4_runtime.neural import NUM_LABELS, DialogueCollator, _device, _loader, _optimizer, _targets, tokenize_dialogues\nelse:\n    from losses import MultilabelLoss, bernoulli_symmetric_kl\n    from selection import StopSelector, better\n    from ver4_runtime.neural import NUM_LABELS, DialogueCollator, _device, _loader, _optimizer, _targets, tokenize_dialogues\n\n\nMODEL_FORMAT = "mission3_ver6_neural"\nLAST_FORMAT = "mission3_ver6_last"\nHEAD_FORMAT = "mission3_ver6_head_trial"\nPROFILES = {\n    "A": {"name": "legacy + ASL", "head_type": "legacy", "loss": "asl", "rdrop": 0.0, "dependency": False},\n    "B": {"name": "legacy + distribution-balanced loss", "head_type": "legacy", "loss": "db", "rdrop": 0.0, "dependency": False},\n    "C": {"name": "legacy + ASL + 0.2 soft macro-F1", "head_type": "legacy", "loss": "asl_soft_f1", "rdrop": 0.0, "dependency": False},\n    "D": {"name": "direct LAAT + weighted BCE", "head_type": "laat", "loss": "bce", "rdrop": 0.0, "dependency": False},\n    "E": {"name": "direct LAAT + weighted BCE + head R-Drop", "head_type": "laat", "loss": "bce", "rdrop": 0.5, "dependency": False},\n    "F": {"name": "direct LAAT + weighted BCE + head R-Drop + dependency", "head_type": "laat", "loss": "bce", "rdrop": 0.5, "dependency": True},\n}\nDEFAULTS = {\n    "head_epochs": 4, "head_batch_size": 32, "head_learning_rate": 3e-4, "head_patience": 2,\n    "profiles": ["A", "B", "C", "D", "E", "F"], "stop_crossfit_splits": 3,\n    "finetune_epochs": 1, "finetune_top_layers": 2, "learning_rate": 2e-6,\n    "finetune_head_learning_rate": 5e-5, "finetune_rdrop": False,\n    "attention_dim": 256, "dropout": 0.2, "batch_size": 2, "window_batch_size": 2,\n    "gradient_accumulation_steps": 16, "inference_batch_size": 8,\n    "gradient_checkpointing": True, "device": "cuda", "log_every": 100,\n    "layerwise_lr_decay": 0.95, "weight_decay": 0.01, "warmup_ratio": 0.1,\n}\n\n\ndef native_bf16_supported():\n    if not torch.cuda.is_available():\n        return False\n    try:\n        return bool(torch.cuda.is_bf16_supported(including_emulation=False))\n    except TypeError:\n        return torch.cuda.get_device_capability()[0] >= 8\n\n\ndef _amp(device):\n    return torch.amp.autocast(device_type=device.type,\n                              dtype=torch.bfloat16 if device.type == "cuda" and native_bf16_supported() else torch.float16,\n                              enabled=device.type == "cuda")\n\n\ndef execution_identity(device):\n    """Bind epoch resumes to one numerical execution environment.\n\n    Reusing a completed pipeline for inference on another accelerator is valid;\n    mixing cached best scores or optimizer/RNG state across GPU precisions is\n    not an exact resume. The pipeline handles completed-output reuse upstream.\n    """\n    device = torch.device(device)\n    identity = {"device_type": device.type, "torch": str(torch.__version__),\n                "transformers": importlib.metadata.version("transformers"),\n                "cuda_version": torch.version.cuda,\n                "amp_dtype": "float32" if device.type != "cuda" else "bfloat16" if native_bf16_supported() else "float16",\n                "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,\n                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32}\n    if device.type == "cuda":\n        identity.update(gpu=torch.cuda.get_device_name(device),\n                        capability=list(torch.cuda.get_device_capability(device)),\n                        cuda_device_count=torch.cuda.device_count())\n    else:\n        identity["cpu_threads"] = torch.get_num_threads()\n    return identity\n\n\nclass LegacyHead(nn.Module):\n    """Exactly the ver4 CLS+mean projection and label-specific window pooling."""\n    def __init__(self, hidden_size, num_labels=NUM_LABELS, dropout=0.2, **unused):\n        super().__init__()\n        self.projection = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden_size))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden_size))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, documents):\n        summaries = [torch.stack([window["legacy_summary"] if "legacy_summary" in window else\n                                  torch.cat([window["hidden"][0], window["hidden"].mean(0)])\n                                  for window in document]) for document in documents]\n        representations = self.dropout(self.projection(pad_sequence(summaries, batch_first=True)))\n        lengths = torch.tensor([len(x) for x in summaries], device=representations.device)\n        mask = torch.arange(representations.shape[1], device=lengths.device)[None, :] < lengths[:, None]\n        scores = (representations @ self.attention_queries.T / math.sqrt(representations.shape[-1])).float()\n        attention = scores.masked_fill(~mask[:, :, None], torch.finfo(scores.dtype).min).softmax(1)\n        pooled = torch.einsum("bwl,bwh->blh", attention.to(representations.dtype), representations)\n        return (self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias\n\n\nclass LabelAttentionHead(nn.Module):\n    """PLM-ICD/LAAT attention over semantic tokens from every window.\n\n    A = softmax(U tanh(W H)); R = A H; each final label logit directly\n    classifies its own original-hidden-size representation. No zero residual\n    bypass exists. Empty semantic input has a finite bias-only prediction.\n    """\n    def __init__(self, hidden_size, num_labels=NUM_LABELS, dropout=0.2,\n                 attention_dim=256, dependency=False, fit_targets=None):\n        super().__init__()\n        self.token_projection = nn.Linear(hidden_size, attention_dim)\n        self.token_queries = nn.Parameter(torch.empty(num_labels, attention_dim))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden_size))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        self.dropout = nn.Dropout(dropout)\n        nn.init.normal_(self.token_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n        self.dependency = bool(dependency)\n        targets = torch.as_tensor(fit_targets, dtype=torch.float32) if fit_targets is not None else None\n        if targets is not None:\n            prior = targets.mean(0).clamp(1e-4, 1 - 1e-4)\n            with torch.no_grad():\n                self.label_bias.copy_(prior.logit())\n        else:\n            prior = torch.full((num_labels,), 0.5)\n        if self.dependency:\n            graph = torch.zeros(num_labels, num_labels)\n            if targets is not None:\n                # graph[source,target] uses FIT-only excess conditional rate.\n                conditional = (targets.T @ targets) / targets.sum(0).clamp_min(1)[:, None]\n                graph = conditional - prior[None, :]\n                graph.fill_diagonal_(0)\n                graph /= graph.abs().sum(0, keepdim=True).clamp_min(1)\n            self.register_buffer("dependency_graph", graph)\n            self.register_buffer("fit_prior", prior)\n            self.dependency_scale = nn.Parameter(torch.full((num_labels,), 0.1))\n\n    def forward(self, documents):\n        rows, lengths = [], []\n        for windows in documents:\n            tokens = torch.cat([window["hidden"][window["content_mask"]] for window in windows], dim=0)\n            lengths.append(len(tokens))\n            # A dummy zero row avoids softmax over a zero-length axis.\n            rows.append(tokens if len(tokens) else windows[0]["hidden"].new_zeros(1, windows[0]["hidden"].shape[-1]))\n        hidden = pad_sequence(rows, batch_first=True)\n        lengths = torch.tensor(lengths, device=hidden.device)\n        mask = torch.arange(hidden.shape[1], device=hidden.device)[None, :] < lengths[:, None]\n        projected = torch.tanh(self.token_projection(self.dropout(hidden)))\n        scores = (projected @ self.token_queries.T).float()\n        attention = scores.masked_fill(~mask[:, :, None], torch.finfo(scores.dtype).min).softmax(1)\n        attention = attention * mask[:, :, None]\n        attention = attention / attention.sum(1, keepdim=True).clamp_min(1e-12)\n        pooled = torch.einsum("btl,bth->blh", attention.to(hidden.dtype), hidden)\n        logits = (self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias\n        if self.dependency:\n            evidence = (logits.sigmoid() - self.fit_prior) @ self.dependency_graph\n            logits = logits + self.dependency_scale.tanh() * evidence\n        return logits\n\n\nclass DialogueClassifier(nn.Module):\n    def __init__(self, encoder, head_type="laat", num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=2, attention_dim=256, special_token_ids=(), dependency=False):\n        super().__init__()\n        self.encoder, self.window_batch_size = encoder, int(window_batch_size)\n        self.special_token_ids = tuple(int(x) for x in special_token_ids)\n        arguments = dict(hidden_size=encoder.config.hidden_size, num_labels=num_labels, dropout=dropout,\n                         attention_dim=attention_dim, dependency=dependency)\n        self.head = LegacyHead(**arguments) if head_type == "legacy" else LabelAttentionHead(**arguments)\n\n    def forward(self, doc_indices, **inputs):\n        documents = [[] for _ in range(int(doc_indices.max().item()) + 1)]\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            subset = {key: value[start:start + self.window_batch_size] for key, value in inputs.items()}\n            hidden = self.encoder(**subset).last_hidden_state\n            for offset in range(len(hidden)):\n                valid = subset["attention_mask"][offset].bool()\n                states = hidden[offset][valid]\n                ids = subset["input_ids"][offset][valid]\n                content = torch.ones(len(ids), dtype=torch.bool, device=ids.device)\n                for token_id in self.special_token_ids:\n                    content &= ids.ne(token_id)\n                documents[int(doc_indices[start + offset])].append({"hidden": states, "content_mask": content})\n        return self.head(documents)\n\n\ndef _read_checkpoint(path):\n    return torch.load(Path(path), map_location="cpu", weights_only=True, mmap=True)\n\n\ndef _encoder(configuration):\n    values = dict(configuration)\n    return AutoModel.from_config(AutoConfig.for_model(values.pop("model_type"), **values))\n\n\ndef _special_ids(tokenizer):\n    return sorted({int(value) for name in ("pad_token_id", "cls_token_id", "sep_token_id", "bos_token_id", "eos_token_id")\n                   if (value := getattr(tokenizer, name, None)) is not None})\n\n\ndef _atomic_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(value, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _settings(config):\n    settings = {name: config.get(name, value) for name, value in DEFAULTS.items()}\n    for name in ("head_epochs", "head_batch_size", "head_patience", "stop_crossfit_splits", "finetune_epochs",\n                 "batch_size", "window_batch_size", "gradient_accumulation_steps", "inference_batch_size", "attention_dim"):\n        minimum = 0 if name == "finetune_epochs" else (2 if name == "stop_crossfit_splits" else 1)\n        if isinstance(settings[name], bool) or not isinstance(settings[name], int) or settings[name] < minimum:\n            raise ValueError(f"{name} must be an integer >= {minimum}")\n    count = settings["finetune_top_layers"]\n    if isinstance(count, bool) or not isinstance(count, int) or count < -1:\n        raise ValueError("finetune_top_layers must be an integer >= -1")\n    for name in ("head_learning_rate", "learning_rate", "finetune_head_learning_rate"):\n        if not math.isfinite(float(settings[name])) or float(settings[name]) <= 0:\n            raise ValueError(f"{name} must be positive and finite")\n    if not 0 <= float(settings["dropout"]) < 1 or not 0 <= float(settings["warmup_ratio"]) < 1:\n        raise ValueError("dropout/warmup_ratio must lie in [0, 1)")\n    if not math.isfinite(float(settings["weight_decay"])) or settings["weight_decay"] < 0:\n        raise ValueError("weight_decay must be nonnegative and finite")\n    profiles = settings["profiles"]\n    if not isinstance(profiles, (list, tuple)) or not profiles or len(set(profiles)) != len(profiles) or any(x not in PROFILES for x in profiles):\n        raise ValueError("profiles must be a nonempty unique sequence chosen from A through F")\n    settings["profiles"] = list(profiles)\n    return settings\n\n\ndef _rng_state(generator):\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(),\n            "numpy": {"name": numpy_state[0], "keys": numpy_state[1].tolist(), "position": int(numpy_state[2]),\n                      "has_gauss": int(numpy_state[3]), "cached_gaussian": float(numpy_state[4])},\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": generator.get_state()}\n\n\ndef _restore_rng(state, generator):\n    random.setstate(state["python"])\n    values = state["numpy"]\n    np.random.set_state((values["name"], np.asarray(values["keys"], dtype=np.uint32), values["position"],\n                         values["has_gauss"], values["cached_gaussian"]))\n    torch.set_rng_state(state["torch"])\n    if state["cuda"]:\n        if not torch.cuda.is_available() or len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("CUDA device count changed; exact epoch resume is unavailable")\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef _seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _cpu_state(module):\n    return {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}\n\n\ndef _check_identity(state, signature, fingerprint, execution=None):\n    if state.get("run_signature") != signature:\n        raise ValueError("Checkpoint run signature mismatch; use a new output directory")\n    if execution is not None and state.get("execution") != execution:\n        raise ValueError("실행 환경/GPU 정밀도가 이전 epoch와 다릅니다. 같은 GPU 종류와 패키지 환경으로 재개하거나 "\n                         "새 OUTPUT_DIR을 사용하세요. Exact epoch resume requires the same execution environment.")\n    if state.get("configuration_fingerprint") != fingerprint:\n        raise ValueError("Checkpoint config fingerprint mismatch; use a new output directory")\n\n\ndef _cached_batch(store, indices, device, legacy=False):\n    documents = []\n    for index in indices:\n        # The legacy head only consumes CLS+mean, so a summary-aware store can\n        # avoid rereading every token shard on every A/B/C epoch.\n        windows = store.get_legacy(int(index)) if legacy and hasattr(store, "get_legacy") else store[int(index)]\n        if not windows:\n            raise ValueError("A feature-cache document must contain windows")\n        documents.append([{"legacy_summary": window["legacy_summary"].to(device=device, dtype=torch.float32)}\n                          if "legacy_summary" in window else\n                          {"hidden": window["hidden"].to(device=device, dtype=torch.float32),\n                           "content_mask": window["content_mask"].to(device=device, dtype=torch.bool)}\n                          for window in windows])\n    return documents\n\n\n@torch.inference_mode()\ndef _predict_head(head, store, indices, batch_size, device):\n    head.eval()\n    predictions = []\n    for start in range(0, len(indices), batch_size):\n        documents = _cached_batch(store, indices[start:start + batch_size], device, legacy=isinstance(head, LegacyHead))\n        with _amp(device):\n            predictions.append(head(documents).float().sigmoid().cpu().numpy())\n    return np.concatenate(predictions)\n\n\ndef _head_from_base(base, profile, settings, fit_targets):\n    hidden = int(base["encoder_config"]["hidden_size"])\n    common = {"hidden_size": hidden, "num_labels": NUM_LABELS, "dropout": settings["dropout"],\n              "attention_dim": settings["attention_dim"], "dependency": profile["dependency"]}\n    if profile["head_type"] == "legacy":\n        head = LegacyHead(**common)\n        head.load_state_dict({key: value for key, value in base["model_state"].items() if not key.startswith("encoder.")}, strict=True)\n    else:\n        head = LabelAttentionHead(**common, fit_targets=fit_targets)\n    return head\n\n\ndef _loss(head_or_model, inputs, labels, criterion, profile, head_phase=True, finetune_rdrop=False):\n    def forward():\n        return head_or_model(inputs) if head_phase else head_or_model(**inputs)\n    first = forward()\n    loss = criterion(first, labels, head_phase=head_phase)\n    rdrop = profile["rdrop"] if head_phase or finetune_rdrop else 0.0\n    if rdrop:\n        second = forward()\n        loss = 0.5 * (loss + criterion(second, labels, head_phase=head_phase)) + rdrop * bernoulli_symmetric_kl(first, second)\n    return loss\n\n\ndef _run_head_trial(profile_id, base, store, fit_indices, stop_indices, targets,\n                    selector, settings, directory, seed, signature, fingerprint, device, execution):\n    profile = PROFILES[profile_id]\n    path = directory / f"profile_{profile_id}.ckpt"\n    state = _read_checkpoint(path) if path.exists() else None\n    if state is not None:\n        if state.get("format") != HEAD_FORMAT or state.get("profile") != profile_id:\n            raise ValueError("Unsupported head trial checkpoint")\n        _check_identity(state, signature, fingerprint, execution)\n        if state["completed"]:\n            return state\n    _seed(seed)\n    head = _head_from_base(base, profile, settings, targets).to(device)\n    generator = torch.Generator().manual_seed(seed)\n    optimizer = torch.optim.AdamW(head.parameters(), lr=settings["head_learning_rate"], weight_decay=settings["weight_decay"])\n    criterion = MultilabelLoss(profile["loss"], targets).to(device)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not native_bf16_supported())\n    completed, history, best_score, best_epoch, stale = 0, [], None, 0, 0\n    if state is not None:\n        head.load_state_dict(state["model_state"])\n        optimizer.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"])\n        _restore_rng(state["rng"], generator)\n        completed, history = state["completed_epochs"], list(state["history"])\n        best_score, best_epoch, stale = state["best_score"], state["best_epoch"], state["stale_epochs"]\n        best_state = state["best_state"]\n    else:\n        started = time.monotonic()\n        best_score = selector.score(_predict_head(head, store, stop_indices, settings["head_batch_size"], device))\n        best_state = _cpu_state(head)\n        history.append({"epoch": 0, "phase": "head_baseline", "train_loss": None, **best_score,\n                        "minutes": (time.monotonic() - started) / 60})\n\n    def commit(epoch, done):\n        result = {"format": HEAD_FORMAT, "format_version": 1, "profile": profile_id,\n                  "execution": execution,\n                  "run_signature": signature, "configuration_fingerprint": fingerprint,\n                  "completed_epochs": epoch, "completed": done, "best_epoch": best_epoch,\n                  "best_score": best_score, "best_state": best_state, "stale_epochs": stale,\n                  "history": history, "model_state": _cpu_state(head), "optimizer": optimizer.state_dict(),\n                  "scaler": scaler.state_dict(), "rng": _rng_state(generator)}\n        _atomic_save(result, path)\n        return result\n\n    if state is None:\n        state = commit(0, False)\n    batch_size = settings["head_batch_size"]\n    for epoch in range(completed + 1, settings["head_epochs"] + 1):\n        started = time.monotonic()\n        head.train()\n        order = torch.randperm(len(fit_indices), generator=generator).numpy()\n        loss_sum = 0.0\n        for start in range(0, len(order), batch_size):\n            rows = order[start:start + batch_size]\n            documents = _cached_batch(store, np.asarray(fit_indices)[rows], device, legacy=profile["head_type"] == "legacy")\n            labels = torch.as_tensor(targets[rows], device=device)\n            optimizer.zero_grad(set_to_none=True)\n            with _amp(device):\n                loss = _loss(head, documents, labels, criterion, profile)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Nonfinite head loss profile={profile_id} epoch={epoch}")\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(head.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            loss_sum += float(loss.detach()) * len(rows)\n            batch_number = start // batch_size + 1\n            if settings["log_every"] > 0 and batch_number % settings["log_every"] == 0:\n                print(f"V6-{profile_id} head epoch={epoch} batch={batch_number}/{math.ceil(len(order) / batch_size)} "\n                      f"loss={loss_sum / (start + len(rows)):.5f} minutes={(time.monotonic() - started) / 60:.1f}", flush=True)\n        score = selector.score(_predict_head(head, store, stop_indices, batch_size, device))\n        history.append({"epoch": epoch, "phase": "head", "train_loss": loss_sum / len(targets), **score,\n                        "minutes": (time.monotonic() - started) / 60})\n        if better(score, best_score):\n            best_score, best_epoch, best_state, stale = score, epoch, _cpu_state(head), 0\n        else:\n            stale += 1\n        state = commit(epoch, epoch == settings["head_epochs"] or stale >= settings["head_patience"])\n        print(f"V6-{profile_id} head epoch={epoch}/{settings[\'head_epochs\']} stop_macro_F1={score[\'stop_macro_f1\']:.6f} "\n              f"stop_AP={score[\'stop_macro_ap\']:.6f} best_epoch={best_epoch} minutes={history[-1][\'minutes\']:.1f}", flush=True)\n        if state["completed"]:\n            break\n    return state\n\n\ndef _finetune_blocks(model, settings):\n    count = settings["finetune_top_layers"]\n    if count <= 0:\n        return []\n    blocks = getattr(getattr(model.encoder, "encoder", None), "layer", None)\n    if not isinstance(blocks, nn.ModuleList) or not len(blocks):\n        raise ValueError("Partial fine-tuning requires BERT/RoBERTa encoder.layer")\n    return list(blocks[-min(count, len(blocks)):])\n\n\ndef _configure_finetune(model, settings):\n    model.encoder.requires_grad_(False)\n    if settings["finetune_top_layers"] == -1:\n        model.encoder.requires_grad_(True)\n    else:\n        for block in _finetune_blocks(model, settings):\n            block.requires_grad_(True)\n    if any(p.requires_grad for p in model.encoder.parameters()) and settings["gradient_checkpointing"]:\n        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    else:\n        model.encoder.gradient_checkpointing_disable()\n    if hasattr(model.encoder.config, "use_cache"):\n        model.encoder.config.use_cache = False\n\n\ndef _train_mode(model, settings):\n    model.train()\n    model.encoder.eval()\n    if settings["finetune_top_layers"] == -1:\n        model.encoder.train()\n    else:\n        for block in _finetune_blocks(model, settings):\n            block.train()\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _from_saved(checkpoint, config):\n    if checkpoint.get("format") not in (MODEL_FORMAT, LAST_FORMAT) or checkpoint.get("format_version") != 1:\n        raise ValueError("Unsupported ver6 neural checkpoint")\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(_encoder(checkpoint["encoder_config"]), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    return model\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    checkpoint = _read_checkpoint(checkpoint_path)\n    if checkpoint.get("format") != MODEL_FORMAT:\n        raise ValueError("Inference requires the selected best ver6 checkpoint")\n    device = _device(config)\n    model = _from_saved(checkpoint, config).to(device)\n    del checkpoint\n    probabilities = _predict(model, _loader(documents, None, tokenizer, config.get("inference_batch_size", 8)), device)\n    if not np.isfinite(probabilities).all():\n        raise FloatingPointError("Inference produced nonfinite probabilities")\n    return probabilities.astype(np.float32, copy=False)\n\n\ndef _best_payload(last):\n    keep = ("format_version", "encoder_config", "architecture", "model_name", "init_epoch", "init_seed",\n            "model_state", "seed", "run_signature", "configuration_fingerprint", "settings", "selected_profile", "execution")\n    return {**{key: last[key] for key in keep}, "format": MODEL_FORMAT, "epoch": last["best_epoch"],\n            "stop_ap": last["best_score"]["stop_macro_ap"], "stop_macro_f1": last["best_score"]["stop_macro_f1"]}\n\n\ndef _result(state, resumed, last_path):\n    return {"best_epoch": state["best_epoch"], "best_stop_ap": state["best_score"]["stop_macro_ap"],\n            "best_stop_macro_f1": state["best_score"]["stop_macro_f1"], "history": state["history"],\n            "completed_epochs": state["completed_epochs"], "resumed": resumed, "last_checkpoint": str(last_path),\n            "selected_profile": state["selected_profile"], "head_trials": state["head_trials"],\n            "execution": state["execution"],\n            "selection_scope": "Original Training fold stop: grouped threshold cross-fit macro-F1; AP secondary. "\n                               "This is reused model/epoch selection, not model-level OOF or official Validation.",\n            "finetune_rdrop": state["settings"]["finetune_rdrop"]}\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets, tokenizer, config,\n                checkpoint_path, seed, init_checkpoint_path, run_signature, feature_store=None,\n                fit_indices=None, stop_indices=None, stop_groups=None):\n    train_targets = _targets(train_targets, train_documents, "fit")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    if feature_store is None or fit_indices is None or stop_indices is None or stop_groups is None:\n        raise ValueError("ver6 requires a shared feature_store, fit_indices, stop_indices, and stop_groups")\n    fit_indices, stop_indices = np.asarray(fit_indices), np.asarray(stop_indices)\n    if len(fit_indices) != len(train_targets) or len(stop_indices) != len(stop_targets):\n        raise ValueError("Feature index order must match supplied fit/stop targets")\n    combined = np.concatenate([fit_indices, stop_indices])\n    if combined.ndim != 1 or not np.issubdtype(combined.dtype, np.integer) or len(np.unique(combined)) != len(combined):\n        raise ValueError("Fit and stop feature indices must be unique and disjoint")\n    if combined.min() < 0 or combined.max() >= len(feature_store):\n        raise ValueError("Feature indices are outside the store")\n    if not isinstance(run_signature, str) or not run_signature:\n        raise ValueError("run_signature must be a nonempty string")\n    settings = _settings(config)\n    device = _device(settings)\n    execution = execution_identity(device)\n    fingerprint_data = {"settings": settings, "seed": int(seed), "initial_checkpoint": str(Path(init_checkpoint_path).resolve()),\n                        "fit_indices": fit_indices.tolist(), "stop_indices": stop_indices.tolist(), "stop_groups": list(map(str, stop_groups)),\n                        "execution": execution}\n    fingerprint = hashlib.sha256(json.dumps(fingerprint_data, sort_keys=True).encode()).hexdigest()\n    checkpoint_path = Path(checkpoint_path)\n    last_path = checkpoint_path.with_name(checkpoint_path.stem + "_last.ckpt")\n    last = _read_checkpoint(last_path) if last_path.exists() else None\n    best_saved = _read_checkpoint(checkpoint_path) if checkpoint_path.exists() else None\n    for saved in (last, best_saved):\n        if saved is not None:\n            _check_identity(saved, run_signature, fingerprint, execution)\n    if last is not None:\n        if last.get("format") != LAST_FORMAT:\n            raise ValueError("Unsupported ver6 resume checkpoint")\n        best_matches = (best_saved is not None and best_saved.get("epoch") == last["best_epoch"] and\n                        best_saved.get("stop_macro_f1") == last["best_score"]["stop_macro_f1"] and\n                        best_saved.get("stop_ap") == last["best_score"]["stop_macro_ap"])\n        if not best_matches:\n            if last["best_epoch"] != last["completed_epochs"]:\n                raise ValueError("Best checkpoint is missing/inconsistent with completed epoch state")\n            _atomic_save(_best_payload(last), checkpoint_path)\n        if last["completed"]:\n            return _result(last, True, last_path)\n    elif best_saved is not None:\n        raise ValueError("Best checkpoint exists without resume state")\n    del best_saved\n    selector = StopSelector(stop_targets, stop_groups, seed=seed, n_splits=settings["stop_crossfit_splits"])\n    base = _read_checkpoint(init_checkpoint_path)\n    if base.get("format_version") != 1 or "model_state" not in base or base.get("format") in (MODEL_FORMAT, LAST_FORMAT):\n        raise ValueError("Warm start requires the original ver4 fold checkpoint")\n    resumed = last is not None or any((checkpoint_path.parent / "head_trials" / f"profile_{profile}.ckpt").exists() for profile in settings["profiles"])\n    trials, winner = [], None\n    for profile_id in settings["profiles"]:\n        state = _run_head_trial(profile_id, base, feature_store, fit_indices, stop_indices, train_targets, selector,\n                                settings, checkpoint_path.parent / "head_trials", seed, run_signature, fingerprint, device, execution)\n        trials.append({"profile": profile_id, **PROFILES[profile_id], "best_epoch": state["best_epoch"],\n                       "best_stop_macro_f1": state["best_score"]["stop_macro_f1"], "best_stop_ap": state["best_score"]["stop_macro_ap"],\n                       "completed_epochs": state["completed_epochs"], "history": state["history"]})\n        if winner is None or better(state["best_score"], winner["best_score"]):\n            winner = state\n    profile_id, profile = winner["profile"], PROFILES[winner["profile"]]\n    print(f"선택: V6-{profile_id}, stop macro-F1={winner[\'best_score\'][\'stop_macro_f1\']:.6f}. 이 후보만 encoder를 미세조정합니다.", flush=True)\n    _seed(seed)\n    if last is None:\n        architecture = {"head_type": profile["head_type"], "num_labels": NUM_LABELS, "dropout": settings["dropout"],\n                        "window_batch_size": settings["window_batch_size"], "attention_dim": settings["attention_dim"],\n                        "special_token_ids": _special_ids(tokenizer), "dependency": profile["dependency"]}\n        model = DialogueClassifier(_encoder(base["encoder_config"]), **architecture)\n        model.encoder.load_state_dict({key.removeprefix("encoder."): value for key, value in base["model_state"].items() if key.startswith("encoder.")}, strict=True)\n        model.head.load_state_dict(winner["best_state"], strict=True)\n        metadata = {"encoder_config": base["encoder_config"], "architecture": architecture,\n                    "model_name": base.get("model_name"), "init_epoch": base.get("epoch"), "init_seed": base.get("seed")}\n    else:\n        if last["selected_profile"] != profile_id:\n            raise ValueError("Selected head changed since partial fine-tune")\n        model = _from_saved(last, settings)\n        metadata = {key: last[key] for key in ("encoder_config", "architecture", "model_name", "init_epoch", "init_seed")}\n    del base, state, winner\n    model.to(device)\n    _configure_finetune(model, settings)\n    loader = _loader(train_documents, train_targets, tokenizer, settings["batch_size"], shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, settings["inference_batch_size"])\n    criterion = MultilabelLoss(profile["loss"], train_targets).to(device)\n    optimizer = _optimizer(model, {**settings, "head_learning_rate": settings["finetune_head_learning_rate"]})\n    total_steps = math.ceil(len(loader) / settings["gradient_accumulation_steps"]) * max(1, settings["finetune_epochs"])\n    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * settings["warmup_ratio"]), total_steps)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not native_bf16_supported())\n    if last is None:\n        started = time.monotonic()\n        # Live encoder evaluation is required: cached fp16 feature rounding can\n        # alter tight threshold ties, so compare live vs live for fine-tuning.\n        best_score = selector.score(_predict(model, stop_loader, device))\n        history = [{"epoch": 0, "phase": "selected_head", "train_loss": None, **best_score,\n                    "minutes": (time.monotonic() - started) / 60}]\n        completed, best_epoch = 0, 0\n    else:\n        optimizer.load_state_dict(last["optimizer"])\n        scheduler.load_state_dict(last["scheduler"])\n        scaler.load_state_dict(last["scaler"])\n        _restore_rng(last["rng"], loader.generator)\n        best_score, best_epoch = last["best_score"], last["best_epoch"]\n        history, completed = list(last["history"]), last["completed_epochs"]\n    del last\n\n    def commit(epoch, done, improved):\n        state = {"format": LAST_FORMAT, "format_version": 1, **metadata, "model_state": _cpu_state(model),\n                 "execution": execution,\n                 "seed": int(seed), "run_signature": run_signature, "configuration_fingerprint": fingerprint,\n                 "settings": settings, "selected_profile": profile_id, "head_trials": trials,\n                 "completed_epochs": epoch, "completed": done, "best_epoch": best_epoch,\n                 "best_score": best_score, "history": history, "optimizer": optimizer.state_dict(),\n                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng": _rng_state(loader.generator)}\n        _atomic_save(state, last_path)\n        if improved:\n            _atomic_save(_best_payload(state), checkpoint_path)\n        return state\n\n    if not last_path.exists():\n        saved = commit(0, settings["finetune_epochs"] == 0, True)\n    else:\n        saved = None\n    accumulation = settings["gradient_accumulation_steps"]\n    for epoch in range(completed + 1, settings["finetune_epochs"] + 1):\n        started = time.monotonic()\n        _train_mode(model, settings)\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, count = 0.0, 0\n        for step, batch in enumerate(loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * loader.batch_size, len(train_documents) - group_start * loader.batch_size)\n            with _amp(device):\n                loss = _loss(model, batch, labels, criterion, profile, head_phase=False, finetune_rdrop=settings["finetune_rdrop"])\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Nonfinite fine-tune loss at epoch {epoch}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += float(loss.detach()) * len(labels)\n            count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            if settings["log_every"] > 0 and (step + 1) % settings["log_every"] == 0:\n                print(f"V6-{profile_id} finetune epoch={epoch} batch={step + 1}/{len(loader)} "\n                      f"loss={loss_sum / count:.5f} minutes={(time.monotonic() - started) / 60:.1f}", flush=True)\n        score = selector.score(_predict(model, stop_loader, device))\n        history.append({"epoch": epoch, "phase": "finetune", "train_loss": loss_sum / count, **score,\n                        "minutes": (time.monotonic() - started) / 60,\n                        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),\n                        "trainable_encoder_parameters": sum(p.numel() for p in model.encoder.parameters() if p.requires_grad),\n                        "total_parameters": sum(p.numel() for p in model.parameters())})\n        improved = better(score, best_score)\n        if improved:\n            best_score, best_epoch = score, epoch\n        saved = commit(epoch, epoch == settings["finetune_epochs"], improved)\n        print(f"V6-{profile_id} finetune epoch={epoch} stop_macro_F1={score[\'stop_macro_f1\']:.6f} "\n              f"stop_AP={score[\'stop_macro_ap\']:.6f} best_finetune_epoch={best_epoch}", flush=True)\n    return _result(saved, resumed, last_path)\n', 'losses.py': '"""Multi-label losses with frequencies estimated from the model-fit split only.\n\nASL follows Ridnik et al.; DB follows the authors\' COCO implementation,\nincluding the divided initialization bias and negative-logit regularization.\nThe soft-F1 hybrid is a simple differentiable macro-F1 adaptation, rather than\na claim to reproduce every setting of the SigmoidF1 paper.\n"""\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef asymmetric_loss(logits, targets, gamma_neg=2.0, gamma_pos=0.0, clip=0.0):\n    probabilities = logits.float().sigmoid()\n    negative = (probabilities - float(clip)).clamp_min(0)\n    positive_term = targets * (1 - probabilities).pow(float(gamma_pos)) * F.logsigmoid(logits.float())\n    # Without clipping, logsigmoid keeps the hard-negative gradient alive even\n    # when sigmoid(z) rounds to 1. With clipping, 1-p+m is bounded below by m.\n    negative_log = F.logsigmoid(-logits.float()) if not clip else torch.log1p(-negative)\n    negative_term = (1 - targets) * negative.pow(float(gamma_neg)) * negative_log\n    return -(positive_term + negative_term).mean()\n\n\ndef soft_macro_f1_loss(logits, targets, epsilon=1e-7):\n    """Compute once over an actual head batch; omit labels absent in that batch.\n\n    BCE/ASL still trains every negative. Omitting unsupported soft-F1 terms\n    avoids a constant zero-gradient term diluting the meaningful class mean.\n    Accumulating gradients of this function is not a larger-batch F1 surrogate.\n    """\n    probabilities = logits.float().sigmoid()\n    support = targets.sum(0)\n    present = support > 0\n    if not bool(present.any()):\n        return logits.sum() * 0.0\n    true_positive = (probabilities * targets).sum(0)\n    scores = (2 * true_positive + epsilon) / (probabilities.sum(0) + support + epsilon)\n    return 1 - scores[present].mean()\n\n\ndef bernoulli_symmetric_kl(first, second):\n    """Mean symmetric KL for independent Bernoulli labels (not label softmax)."""\n    first, second = first.float(), second.float()\n    p, q = first.sigmoid(), second.sigmoid()\n    log_p, log_q = F.logsigmoid(first), F.logsigmoid(second)\n    log_np, log_nq = F.logsigmoid(-first), F.logsigmoid(-second)\n    pq = p * (log_p - log_q) + (1 - p) * (log_np - log_nq)\n    qp = q * (log_q - log_p) + (1 - q) * (log_nq - log_np)\n    return 0.5 * (pq + qp).mean()\n\n\nclass DistributionBalancedLoss(nn.Module):\n    """DB rebalancing + negative tolerant regularization + focal weighting.\n\n    Original COCO constants are used with ordinary shuffled document batches;\n    this sampling choice is an explicit adaptation. An all-negative row uses\n    the limiting rebalance weight alpha+1, avoiding division by zero.\n    """\n    def __init__(self, fit_targets, alpha=0.1, beta=10.0, mu=0.2,\n                 negative_scale=2.0, initial_bias=0.05, gamma=2.0, balance=2.0):\n        super().__init__()\n        targets = torch.as_tensor(fit_targets, dtype=torch.float32)\n        if targets.ndim != 2 or not len(targets) or not torch.isin(targets, torch.tensor([0., 1.])).all():\n            raise ValueError("DB fit_targets must be a nonempty binary matrix")\n        counts = targets.sum(0)\n        if bool((counts <= 0).any()):\n            raise ValueError("DB requires at least one positive per fit label")\n        if negative_scale <= 0:\n            raise ValueError("negative_scale must be positive")\n        frequency = (counts / len(targets)).clamp(1e-6, 1 - 1e-6)\n        self.register_buffer("inverse_frequency", counts.reciprocal())\n        self.register_buffer("initial_bias", frequency.logit() * float(initial_bias) / float(negative_scale))\n        self.alpha, self.beta, self.mu = float(alpha), float(beta), float(mu)\n        self.negative_scale, self.gamma, self.balance = float(negative_scale), float(gamma), float(balance)\n\n    def forward(self, logits, targets):\n        denominator = (targets * self.inverse_frequency).sum(1, keepdim=True)\n        ratio = self.inverse_frequency / denominator.clamp_min(1e-12)\n        rebalance = torch.sigmoid(self.beta * (ratio - self.mu)) + self.alpha\n        rebalance = torch.where(denominator > 0, rebalance, torch.full_like(rebalance, 1 + self.alpha))\n        scale = targets + (1 - targets) * self.negative_scale\n        normalized_logits = (logits.float() + self.initial_bias) * scale\n        bce = F.binary_cross_entropy_with_logits(normalized_logits, targets, reduction="none")\n        focal = (1 - torch.exp(-bce)).pow(self.gamma)\n        weights = rebalance * (targets + (1 - targets) / self.negative_scale)\n        return (self.balance * focal * weights * bce).mean()\n\n\nclass MultilabelLoss(nn.Module):\n    def __init__(self, name, fit_targets, soft_f1_weight=0.2):\n        super().__init__()\n        targets = torch.as_tensor(fit_targets, dtype=torch.float32)\n        counts = targets.sum(0)\n        if bool((counts <= 0).any()):\n            raise ValueError("Every fit label needs positive examples")\n        self.name, self.soft_f1_weight = name, float(soft_f1_weight)\n        weight = (((len(targets) - counts) / counts).clamp_min(1).sqrt()).clamp(1, 5)\n        self.register_buffer("pos_weight", weight)\n        self.db = DistributionBalancedLoss(targets) if name == "db" else None\n        if name not in ("bce", "asl", "asl_soft_f1", "db"):\n            raise ValueError(f"Unknown loss: {name}")\n\n    def forward(self, logits, targets, head_phase=True):\n        targets = targets.float()\n        if self.name == "bce":\n            return F.binary_cross_entropy_with_logits(logits.float(), targets, pos_weight=self.pos_weight)\n        if self.name == "db":\n            return self.db(logits, targets)\n        loss = asymmetric_loss(logits, targets)\n        if self.name == "asl_soft_f1" and head_phase:\n            loss = loss + self.soft_f1_weight * soft_macro_f1_loss(logits, targets)\n        return loss\n', 'selection.py': '"""Grouped threshold cross-fitting on the original fold\'s stopping split.\n\nThis is model/epoch selection on Training stop. It is not model-level OOF and\nits repeatedly selected score is not an unbiased final performance estimate.\nNo official Validation or model-calibration rows belong here.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, f1_score\n\nif __package__:\n    from .ver4_runtime.calibration import _best_threshold, _grouped_splits\nelse:\n    from ver4_runtime.calibration import _best_threshold, _grouped_splits\n\n\nclass StopSelector:\n    def __init__(self, targets, groups, seed=44, n_splits=3):\n        self.targets = np.asarray(targets, dtype=np.float32)\n        if self.targets.ndim != 2 or not len(self.targets) or not np.isin(self.targets, [0, 1]).all():\n            raise ValueError("Stop targets must be a nonempty binary matrix")\n        self.groups = np.asarray(groups)\n        self.folds = list(_grouped_splits(self.targets, self.groups, seed, n_splits))\n\n    def score(self, probabilities):\n        probabilities = np.asarray(probabilities, dtype=np.float64)\n        if probabilities.shape != self.targets.shape or not np.isfinite(probabilities).all():\n            raise ValueError("Stop probabilities must be finite and match targets")\n        if np.any((probabilities < 0) | (probabilities > 1)):\n            raise ValueError("Stop probabilities must lie in [0, 1]")\n        predictions = np.zeros_like(self.targets, dtype=np.int8)\n        seen = np.zeros(len(self.targets), dtype=np.int8)\n        thresholds = []\n        for fit, score in self.folds:\n            threshold = [_best_threshold(self.targets[fit, label], probabilities[fit, label])[0]\n                         for label in range(self.targets.shape[1])]\n            predictions[score] = probabilities[score] >= np.asarray(threshold)\n            seen[score] += 1\n            thresholds.append(threshold)\n        if not np.all(seen == 1):\n            raise RuntimeError("Every stop row must be scored once")\n        present = self.targets.sum(0) > 0\n        ap = float(average_precision_score(self.targets[:, present], probabilities[:, present], average="macro")) if present.any() else 0.0\n        return {"stop_macro_f1": float(f1_score(self.targets, predictions, average="macro", zero_division=0)),\n                "stop_macro_ap": ap, "stop_labels_with_positives": int(present.sum()),\n                "threshold_crossfit_splits": len(self.folds), "thresholds_by_fold": thresholds}\n\n\ndef better(score, best, tolerance=1e-6):\n    """F1 selects; AP is only a deterministic secondary criterion for F1 ties."""\n    if best is None:\n        return True\n    delta = score["stop_macro_f1"] - best["stop_macro_f1"]\n    return delta > tolerance or (abs(delta) <= tolerance and score["stop_macro_ap"] > best["stop_macro_ap"] + tolerance)\n', 'features.py': '"""One frozen encoder pass, reused by every head trial; no labels in the cache.\n\nCompleted shards are atomic and content-verified. A new session rebuilds only\nmissing shards; repeated trials mmap local float16 features without loading the\nwhole corpus into RAM. CLS/SEP are retained for the exact legacy pooling path.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import nullcontext\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom transformers import AutoConfig, AutoModel\n\n\ndef native_bf16_supported():\n    if not torch.cuda.is_available():\n        return False\n    try:\n        return bool(torch.cuda.is_bf16_supported(including_emulation=False))\n    except TypeError:\n        return torch.cuda.get_device_capability()[0] >= 8\n\n\ndef amp_context(device):\n    device = torch.device(device)\n    if device.type != "cuda":\n        return nullcontext()\n    return torch.amp.autocast("cuda", dtype=torch.bfloat16 if native_bf16_supported() else torch.float16)\n\n\ndef _hash(path):\n    value = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\ndef _write_json(path, value):\n    temporary = path.with_suffix(".tmp")\n    temporary.write_text(_json(value), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _encoder(path):\n    checkpoint = torch.load(path, map_location="cpu", weights_only=True, mmap=True)\n    config = dict(checkpoint["encoder_config"])\n    model_type = config.pop("model_type")\n    # Lazy extraction can run after restoring a head\'s RNG state. Constructing\n    # a temporary encoder must not perturb that state\'s subsequent dropout.\n    with torch.random.fork_rng(devices=[]):\n        encoder = AutoModel.from_config(AutoConfig.for_model(model_type, **config))\n    state = {key.removeprefix("encoder."): value for key, value in checkpoint["model_state"].items()\n             if key.startswith("encoder.")}\n    if not state:\n        raise ValueError("Base checkpoint contains no encoder")\n    encoder.load_state_dict(state, strict=True)\n    encoder.requires_grad_(False)\n    encoder.eval()\n    return encoder\n\n\nclass FeatureStore:\n    """Sequence of documents, each a list of hidden/content_mask window dicts."""\n\n    def __init__(self, directory, documents, special_ids, shards, identity):\n        self.directory = Path(directory)\n        self.documents = documents\n        self.special_ids = set(special_ids)\n        self.shards = shards\n        self.identity = identity\n        self.arrays = {}\n        self.legacy_summaries = {}\n        self.lookup = []\n        for number, shard in enumerate(shards):\n            cursor = 0\n            for index in range(shard["start"], shard["end"]):\n                windows = []\n                for window in documents[index]:\n                    length = len(window["input_ids"])\n                    windows.append((cursor, cursor + length))\n                    cursor += length\n                self.lookup.append((number, windows))\n            if cursor != shard["shape"][0]:\n                raise ValueError("Feature shard token offsets disagree with input")\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        if not isinstance(index, (int, np.integer)):\n            raise TypeError("FeatureStore indices must be integers")\n        if index < 0:\n            index += len(self)\n        if not 0 <= index < len(self):\n            raise IndexError(index)\n        number, offsets = self.lookup[index]\n        if number not in self.arrays:\n            path = self.directory / self.shards[number]["file"]\n            self.arrays[number] = np.load(path, mmap_mode="r", allow_pickle=False)\n        values = self.arrays[number]\n        result = []\n        for window, (start, end) in zip(self.documents[index], offsets):\n            ids = window["input_ids"]\n            attention = window.get("attention_mask", [1] * len(ids))\n            content = [bool(active) and token not in self.special_ids for token, active in zip(ids, attention)]\n            # Copies only one window, avoiding writable aliases into the cache.\n            result.append({"hidden": torch.from_numpy(np.array(values[start:end], copy=True)),\n                           "content_mask": torch.tensor(content, dtype=torch.bool)})\n        return result\n\n    def get_legacy(self, index):\n        """Keep only CLS+mean in RAM for A-C, avoiding repeated full-token I/O."""\n        if not isinstance(index, (int, np.integer)):\n            raise TypeError("FeatureStore indices must be integers")\n        if index < 0:\n            index += len(self)\n        if not 0 <= index < len(self):\n            raise IndexError(index)\n        if index not in self.legacy_summaries:\n            summaries = []\n            for window in self[index]:\n                hidden = window["hidden"].float()\n                summaries.append(torch.cat([hidden[0], hidden.mean(0)]))\n            self.legacy_summaries[index] = summaries\n        return [{"legacy_summary": value.clone()} for value in self.legacy_summaries[index]]\n\n    def close(self):\n        self.arrays.clear()\n        self.legacy_summaries.clear()\n\n\ndef build_feature_store(documents, tokenizer, checkpoint_path, cache_dir, config,\n                        checkpoint_sha256=None, shard_documents=256):\n    if not documents or any(not windows for windows in documents):\n        raise ValueError("Features require nonempty documents/windows")\n    if type(shard_documents) is not int or shard_documents < 1:\n        raise ValueError("shard_documents must be positive")\n    device = torch.device(config.get("device", "cpu"))\n    batch_size = int(config.get("window_batch_size", 2))\n    if batch_size < 1:\n        raise ValueError("window_batch_size must be positive")\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA feature extraction requires a GPU runtime")\n    special_ids = sorted({int(value) for name in ("pad_token_id", "cls_token_id", "sep_token_id",\n                                                  "bos_token_id", "eos_token_id")\n                          if (value := getattr(tokenizer, name, None)) is not None})\n    execution = {"device": str(device), "torch": torch.__version__,\n                 "transformers": importlib.metadata.version("transformers"),\n                 "window_batch_size": batch_size, "stored_dtype": "float16"}\n    if device.type == "cuda":\n        execution.update(gpu=torch.cuda.get_device_name(device),\n                         capability=list(torch.cuda.get_device_capability(device)),\n                         native_bf16=native_bf16_supported(), cuda_version=torch.version.cuda)\n    identity = {"format": "mission3_frozen_features_v1", "execution": execution,\n                "weights": checkpoint_sha256 or _hash(checkpoint_path),\n                "inputs": hashlib.sha256(_json(documents).encode()).hexdigest(),\n                "special_ids": special_ids, "shard_documents": shard_documents,\n                "code": _hash(__file__)}\n    signature = hashlib.sha256(_json(identity).encode()).hexdigest()\n    directory = Path(cache_dir).expanduser().resolve() / signature\n    directory.mkdir(parents=True, exist_ok=True)\n    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True, mmap=True)\n    hidden_size = int(saved["encoder_config"]["hidden_size"])\n    del saved\n    shards, pending = [], []\n    for number, start in enumerate(range(0, len(documents), shard_documents)):\n        end = min(start + shard_documents, len(documents))\n        tokens = sum(len(window["input_ids"]) for doc in documents[start:end] for window in doc)\n        record = {"start": start, "end": end, "shape": [tokens, hidden_size], "file": f"{number:05d}.npy"}\n        path = directory / record["file"]\n        meta = path.with_suffix(".json")\n        valid = False\n        try:\n            previous = json.loads(meta.read_text())\n            if all(previous.get(key) == value for key, value in record.items()):\n                stat = path.stat()\n                valid = stat.st_size == previous["size"]\n                if valid and stat.st_mtime_ns != previous["mtime_ns"]:\n                    valid = _hash(path) == previous["sha256"]\n                if valid:\n                    array = np.load(path, mmap_mode="r", allow_pickle=False)\n                    valid = list(array.shape) == record["shape"] and array.dtype == np.float16\n                    del array\n        except (OSError, ValueError, KeyError, EOFError):\n            pass\n        shards.append(record)\n        if not valid:\n            pending.append(number)\n    required = sum(np.prod(shards[n]["shape"], dtype=np.int64) * 2 + 256 for n in pending)\n    if required > shutil.disk_usage(directory).free:\n        raise OSError(f"Frozen feature cache needs {required / 2**30:.2f} GiB of free LOCAL disk space")\n    total_bytes = sum(np.prod(s["shape"], dtype=np.int64) * 2 for s in shards)\n    print(f"동결 특징: {len(documents):,}개 통화 / 약 {total_bytes / 2**30:.2f} GiB / "\n          f"재사용 {len(shards)-len(pending)}/{len(shards)} 조각", flush=True)\n    encoder = None\n    started = time.monotonic()\n    try:\n        if pending:\n            encoder = _encoder(checkpoint_path).to(device).eval()\n        for number in pending:\n            record = shards[number]\n            path = directory / record["file"]\n            temporary = None\n            try:\n                with tempfile.NamedTemporaryFile(dir=directory, suffix=".tmp", delete=False) as f:\n                    temporary = Path(f.name)\n                array = np.lib.format.open_memmap(temporary, mode="w+", dtype=np.float16,\n                                                 shape=tuple(record["shape"]))\n                windows = [window for doc in documents[record["start"]:record["end"]] for window in doc]\n                cursor = 0\n                for start in range(0, len(windows), batch_size):\n                    chunk = windows[start:start + batch_size]\n                    inputs = tokenizer.pad(chunk, padding=True, return_tensors="pt")\n                    inputs = {key: value.to(device) for key, value in inputs.items()}\n                    with torch.inference_mode(), amp_context(device):\n                        hidden = encoder(**inputs).last_hidden_state\n                    for i, window in enumerate(chunk):\n                        length = len(window["input_ids"])\n                        values = hidden[i, :length].float().cpu().numpy().astype(np.float16)\n                        if not np.isfinite(values).all():\n                            raise FloatingPointError("Non-finite frozen token features")\n                        array[cursor:cursor + length] = values\n                        cursor += length\n                if cursor != record["shape"][0]:\n                    raise RuntimeError("Feature extraction lost token rows")\n                array.flush()\n                del array\n                digest = _hash(temporary)\n                temporary.replace(path)\n                stat = path.stat()\n                _write_json(path.with_suffix(".json"), {**record, "sha256": digest,\n                            "size": stat.st_size, "mtime_ns": stat.st_mtime_ns})\n                print(f"동결 특징 저장 {number+1}/{len(shards)}: "\n                      f"통화 {record[\'end\']:,}/{len(documents):,}, {(time.monotonic()-started)/60:.1f}분", flush=True)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n    finally:\n        del encoder\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n    _write_json(directory / "index.json", {"identity": identity, "signature": signature,\n                "documents": len(documents), "size_bytes": int(total_bytes), "shards": shards})\n    return FeatureStore(directory, documents, special_ids, shards, identity)\n', 'calibration.py': '"""Training-only selection between ver4 and two bounded three-way mixtures."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from .ver4_runtime.data import TARGET_SYMPTOMS\nelse:\n    from ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from ver4_runtime.data import TARGET_SYMPTOMS\n\nCANDIDATE_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nLOGIT_EPSILON = 1e-6\n\n\ndef _blend_space(value):\n    if value not in ("raw", "logit"):\n        raise ValueError("blend_space must be \'raw\' or \'logit\'")\n    return value\n\n\ndef _logits(probabilities):\n    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64),\n                            LOGIT_EPSILON, 1.0 - LOGIT_EPSILON)\n    return np.log(probabilities) - np.log1p(-probabilities)\n\n\ndef _sigmoid(logits):\n    # Convex mixtures of clipped logits stay within finite, safe exp bounds.\n    return 1.0 / (1.0 + np.exp(-logits))\n\n\ndef _validated_inputs(y_true, base_neural, sparse, candidate):\n    truth, base, sparse = _inputs(y_true, base_neural, sparse)\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != base.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    return truth, base, sparse, candidate\n\n\ndef _with_zero_candidate(calibrator):\n    return {**calibrator, "candidate_weights": [0.0] * len(TARGET_SYMPTOMS),\n            "blend_space": "raw"}\n\n\ndef _fit_mixture(y_true, base_neural, sparse, candidate, blend_space="raw"):\n    """Exact F1 selection per label; equal scores prefer zero candidate weight."""\n    blend_space = _blend_space(blend_space)\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    if blend_space == "logit":\n        base, sparse, candidate = map(_logits, (base, sparse, candidate))\n    alphas, betas, thresholds = [], [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        best = None\n        for alpha in WEIGHT_GRID:\n            baseline = alpha * base[:, column] + (1.0 - alpha) * sparse[:, column]\n            for beta in CANDIDATE_GRID:\n                probabilities = (1.0 - beta) * baseline + beta * candidate[:, column]\n                if blend_space == "logit":\n                    probabilities = _sigmoid(probabilities)\n                threshold, score = _best_threshold(truth[:, column], probabilities)\n                # Keep ver4\'s alpha tie convention after preferring smaller beta.\n                choice = (score, -beta, -abs(alpha - 0.5), alpha, threshold)\n                if best is None or choice > best:\n                    best = choice\n        _, negative_beta, _, alpha, threshold = best\n        alphas.append(float(alpha))\n        betas.append(float(-negative_beta))\n        thresholds.append(float(threshold))\n    return {"neural_weights": alphas, "candidate_weights": betas,\n            "thresholds": thresholds, "blend_space": blend_space}\n\n\ndef apply_calibrator(base_neural, sparse, candidate, calibrator):\n    """Apply the stored space; older manifests default to the exact raw blend."""\n    blend_space = _blend_space(calibrator.get("blend_space", "raw"))\n    # Reuse ver4\'s shape, range, alpha and threshold validation in both spaces.\n    baseline = _apply_baseline(base_neural, sparse, calibrator)\n    betas = np.asarray(calibrator["candidate_weights"], dtype=np.float64)\n    if betas.shape != (len(TARGET_SYMPTOMS),) or not np.isfinite(betas).all():\n        raise ValueError("candidate_weights must contain one finite weight per symptom")\n    if np.any((betas < 0) | (betas > 1)):\n        raise ValueError("candidate_weights must be in [0, 1]")\n    if blend_space == "logit":\n        alphas = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n        baseline = alphas * _logits(base_neural) + (1.0 - alphas) * _logits(sparse)\n    if candidate is None:\n        if np.any(betas != 0):\n            raise ValueError("candidate probabilities are required for nonzero candidate_weights")\n        return _sigmoid(baseline) if blend_space == "logit" else baseline\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != baseline.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    if blend_space == "logit":\n        candidate = _logits(candidate)\n    mixture = (1.0 - betas) * baseline + betas * candidate\n    return _sigmoid(mixture) if blend_space == "logit" else mixture\n\n\ndef _metrics(truth, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, predictions, average=None, zero_division=0,\n    )\n    return {\n        symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n        for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n    }\n\n\ndef calibrate_crossfit(y_true, base_neural, sparse, candidate, groups, seed=44, n_splits=5,\n                       min_gain=0.002, min_improved_folds=3):\n    """Tune on calibration-fold training rows, compare on excluded groups.\n\n    All supplied probabilities must originate from base/candidate models that\n    excluded this entire Training calibration set from model fitting and epoch\n    selection. These arrays must never contain official Validation samples.\n    Compare only the original baseline and fixed raw/logit mixture families.\n    The global procedure choice uses the resulting cross-fit scores, so its\n    selected score is a selection estimate, not an unbiased performance guarantee.\n    """\n    if not np.isfinite(min_gain) or min_gain < 0:\n        raise ValueError("min_gain must be finite and nonnegative")\n    if type(min_improved_folds) is not int or min_improved_folds < 1:\n        raise ValueError("min_improved_folds must be a positive integer")\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    baseline_predictions = np.zeros_like(truth)\n    mixture_predictions = {space: np.zeros_like(truth) for space in ("raw", "logit")}\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    fold_gains = {space: [] for space in ("raw", "logit")}\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        baseline_calibrator = _fit_baseline(truth[train], base[train], sparse[train])\n        baseline_predictions[test] = (\n            _apply_baseline(base[test], sparse[test], baseline_calibrator)\n            >= np.asarray(baseline_calibrator["thresholds"])\n        )\n        fold_baseline_score = f1_score(truth[test], baseline_predictions[test], average="macro", zero_division=0)\n        for space, predictions in mixture_predictions.items():\n            mixture_calibrator = _fit_mixture(\n                truth[train], base[train], sparse[train], candidate[train], blend_space=space,\n            )\n            predictions[test] = (\n                apply_calibrator(base[test], sparse[test], candidate[test], mixture_calibrator)\n                >= np.asarray(mixture_calibrator["thresholds"])\n            )\n            fold_gains[space].append(float(f1_score(truth[test], predictions[test], average="macro", zero_division=0)\n                                          - fold_baseline_score))\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n    baseline_score = float(f1_score(truth, baseline_predictions, average="macro", zero_division=0))\n    mixture_scores = {\n        space: float(f1_score(truth, predictions, average="macro", zero_division=0))\n        for space, predictions in mixture_predictions.items()\n    }\n    # A single global family choice avoids per-symptom selection on scored folds.\n    candidate_space = "logit" if mixture_scores["logit"] > mixture_scores["raw"] else "raw"\n    candidate_score = mixture_scores[candidate_space]\n    candidate_predictions = mixture_predictions[candidate_space]\n    baseline_final = _with_zero_candidate(_fit_baseline(truth, base, sparse))\n    gain = candidate_score - baseline_score\n    required_folds = min(min_improved_folds, len(fold_counts))\n    improved_folds = sum(value > 0 for value in fold_gains[candidate_space])\n    material = candidate_score > baseline_score and gain >= min_gain\n    consistent = improved_folds >= required_folds\n    use_candidate = material and consistent\n    selected = (_fit_mixture(truth, base, sparse, candidate, blend_space=candidate_space)\n                if use_candidate else baseline_final)\n    selected_predictions = candidate_predictions if use_candidate else baseline_predictions\n    fitted = apply_calibrator(base, sparse, candidate, selected) >= np.asarray(selected["thresholds"])\n    report = {\n        "score_scope": "Training-only calibration. Procedure selection reuses cross-fit scores; selected score is a selection estimate, not an unbiased guarantee or official Validation.",\n        "baseline_crossfit_macro_f1": baseline_score,\n        "candidate_crossfit_macro_f1": candidate_score,\n        "candidate_raw_crossfit_macro_f1": mixture_scores["raw"],\n        "candidate_logit_crossfit_macro_f1": mixture_scores["logit"],\n        "candidate_blend_space": candidate_space,\n        "selected_blend_space": selected["blend_space"],\n        "selected_crossfit_macro_f1": candidate_score if use_candidate else baseline_score,\n        "selected_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "selected_name": "candidate_mixture" if use_candidate else "ver4_baseline",\n        "fallback_reason": (None if use_candidate else\n                            "candidate_gain_below_minimum" if not material else "candidate_gain_not_consistent_across_folds"),\n        "selection_gain": float(gain), "minimum_gain": float(min_gain),\n        "candidate_improved_folds": improved_folds, "minimum_improved_folds": required_folds,\n        "candidate_fold_gains": fold_gains[candidate_space], "all_candidate_fold_gains": fold_gains,\n        "per_symptom_crossfit": {\n            "baseline": _metrics(truth, baseline_predictions),\n            "candidate": _metrics(truth, candidate_predictions),\n            "candidate_raw": _metrics(truth, mixture_predictions["raw"]),\n            "candidate_logit": _metrics(truth, mixture_predictions["logit"]),\n            "selected": _metrics(truth, selected_predictions),\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n        "selection_rule": "Compare fixed raw/logit families globally; ties prefer raw. Require strictly positive aggregate gain at least minimum_gain and improvements on minimum_improved_folds; otherwise use ver4. Training-only selection guard, not a statistical significance test or Validation guarantee.",\n    }\n    return selected, report\n', 'prediction_cache.py': '"""Reuse raw inference probabilities across experiments without reading labels.\n\nKeys bind ordered inputs, weights, tokenizer, inference code and execution\nsettings. Calibration/thresholds are deliberately excluded: they are applied\nafter loading raw probabilities. A missing or damaged cache is recomputed.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nfrom zipfile import BadZipFile\n\nimport numpy as np\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\nclass PredictionCache:\n    def __init__(self, directory, paths, texts, tokenizer_dir, labels, execution):\n        self.directory = Path(directory).expanduser().resolve()\n        self.shape = (len(paths), len(labels))\n        if len(paths) != len(texts):\n            raise ValueError("Cache paths/texts length mismatch")\n        tokenizer_dir = Path(tokenizer_dir)\n        self.context = {\n            "format": "mission3_raw_probabilities_v1",\n            "inputs": hashlib.sha256(_json([\n                [Path(path).name, text] for path, text in zip(paths, texts)\n            ]).encode("utf-8")).hexdigest(),\n            "labels": list(labels), "execution": execution,\n            "tokenizer": {p.relative_to(tokenizer_dir).as_posix(): _file_hash(p)\n                          for p in sorted(tokenizer_dir.rglob("*")) if p.is_file()},\n        }\n\n    def _valid(self, values):\n        return (values.shape == self.shape and np.isfinite(values).all()\n                and np.all((values >= 0) & (values <= 1)))\n\n    def get(self, kind, checkpoint, config, sources, compute):\n        # Do not bind absolute paths: a copied identical bundle can reuse results.\n        identity = _json({\n            **self.context, "kind": kind, "weights": _file_hash(checkpoint),\n            "settings": {key: config[key] for key in\n                         ("device", "max_length", "stride", "window_batch_size", "inference_batch_size")},\n            "code": [_file_hash(path) for path in sources],\n        })\n        key = hashlib.sha256(identity.encode("utf-8")).hexdigest()\n        path = self.directory / (key + ".npz")\n        if path.is_file():\n            try:\n                with np.load(path, allow_pickle=False) as stored:\n                    values = np.array(stored["probabilities"], dtype=np.float64)\n                    checksum = hashlib.sha256(values.tobytes()).hexdigest()\n                    if (stored["identity"].item() == identity and self._valid(values)\n                            and stored["checksum"].item() == checksum):\n                        print(f"예측 캐시 재사용: {kind} / {Path(checkpoint).name}", flush=True)\n                        return values\n            except (OSError, ValueError, KeyError, EOFError, BadZipFile):\n                pass\n            print(f"예측 캐시 손상: {kind}, 다시 계산합니다.", flush=True)\n        values = np.asarray(compute(), dtype=np.float64)\n        if not self._valid(values):\n            raise ValueError("Invalid raw probabilities; cache was not written")\n        self.directory.mkdir(parents=True, exist_ok=True)\n        # Unique temporary files keep interrupted/concurrent writes out of reads.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(dir=self.directory, suffix=".tmp", delete=False) as stream:\n                temporary = Path(stream.name)\n                np.savez_compressed(stream, identity=np.asarray(identity), probabilities=values,\n                                    checksum=np.asarray(hashlib.sha256(values.tobytes()).hexdigest()))\n            temporary.replace(path)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n        print(f"예측 캐시 저장: {kind} / {Path(checkpoint).name}", flush=True)\n        return values\n', 'pipeline.py': '"""Reuse ver4; fit one complementary model without touching official Validation."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from . import calibration, features, neural\n    from .prediction_cache import PredictionCache\n    from .ver4_runtime import pipeline as base_pipeline\n    from .ver4_runtime import neural as base_neural\n    from .ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\nelse:\n    import calibration\n    import features\n    import neural\n    from prediction_cache import PredictionCache\n    from ver4_runtime import pipeline as base_pipeline\n    from ver4_runtime import neural as base_neural\n    from ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group, write_json, write_predictions\n\n\nDEFAULT_CONFIG = {\n    **neural.DEFAULTS, "seed": 42, "init_fold": 1, "require_cuda": True,\n    "bundle_base": False, "feature_cache_dir": None,\n    "min_calibration_gain": 0.002, "min_calibration_improved_folds": 3,\n}\nRUNTIME_FILES = (\n    "__init__.py", "pipeline.py", "neural.py", "losses.py", "selection.py", "features.py",\n    "calibration.py", "prediction_cache.py", "train.py", "inference.py",\n    "bootstrap.py", "requirements.txt", "README.md", "RESEARCH.md",\n)\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _atomic_torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _release():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\nclass _LazyFeatureStore:\n    """Completed head trials can resume fine-tuning without rebuilding features."""\n\n    def __init__(self, size, directory, factory):\n        self.size, self.directory, self.factory = size, Path(directory).expanduser().resolve(), factory\n        self.store, self.minutes = None, 0.0\n\n    def __len__(self):\n        return self.size\n\n    def _materialize(self):\n        if self.store is None:\n            started = time.monotonic()\n            self.store = self.factory()\n            self.minutes = (time.monotonic() - started) / 60\n            self.directory = self.store.directory\n        return self.store\n\n    def __getitem__(self, index):\n        return self._materialize()[index]\n\n    def get_legacy(self, index):\n        return self._materialize().get_legacy(index)\n\n    def report(self):\n        return {"directory": str(self.directory), "documents": self.size, "minutes": self.minutes,\n                "features_accessed_this_run": self.store is not None,\n                "scope": "original fold fit+stop inputs only; no labels in cache"}\n\n    def close(self):\n        if self.store is not None:\n            self.store.close()\n\n\ndef _versions():\n    return {name: importlib.metadata.version(name) for name in\n            ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n\n\ndef _check_versions(base_manifest):\n    current = _versions()\n    # sklearn objects in sparse.joblib require the original compatible environment.\n    for name in ("transformers", "numpy", "scikit-learn"):\n        saved = base_manifest.get("versions", {}).get(name)\n        if saved and current[name] != saved:\n            raise RuntimeError(f"{name}: ver4 저장 버전={saved}, 현재={current[name]}. requirements.txt를 설치하세요.")\n    return current\n\n\ndef _read_base(train_label_dir, base_dir, init_fold):\n    """Verify the exact original training run before trusting cached predictions."""\n    checkpoint = base_dir / "mission3_ver4_model.ckpt"\n    manifest = base_pipeline.load_manifest(checkpoint)\n    _check_versions(manifest)\n    print("기존 ver4 확인 완료. 서울 Training JSON을 읽습니다...", flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    print(f"Training {len(paths):,}개 로드 완료. 원래 데이터·분할·보정 캐시를 검증합니다...", flush=True)\n    fingerprint = base_pipeline._fingerprint(paths, texts, targets, manifest["config"])\n    if fingerprint != manifest.get("signature"):\n        raise ValueError("ver4와 Training 데이터/전처리/코드가 다릅니다. 원래 Training과 ver4 결과 묶음을 사용하세요.")\n    splits = json.loads((base_dir / "splits.json").read_text(encoding="utf-8"))\n    lookup = {p.name: i for i, p in enumerate(paths)}\n    all_names = set(lookup)\n    group_ids = np.asarray([text_group(text) for text in texts])\n\n    def indices(names, purpose):\n        if not names or len(names) != len(set(names)) or not set(names) <= all_names:\n            raise ValueError(f"Invalid saved {purpose} filenames")\n        return np.asarray([lookup[name] for name in names], dtype=np.int64)\n\n    cal = indices(splits["calibration"], "calibration")\n    if len(splits["folds"]) != len(manifest["folds"]) or not 1 <= init_fold <= len(splits["folds"]):\n        raise ValueError("init_fold / saved folds mismatch")\n    fold_indices = []\n    for number, fold in enumerate(splits["folds"], 1):\n        fit, stop = indices(fold["fit"], "fit"), indices(fold["stop"], "stop")\n        sets = [set(fit), set(stop), set(cal)]\n        groups = [set(group_ids[x]) for x in (fit, stop, cal)]\n        if any(sets[i] & sets[j] or groups[i] & groups[j] for i, j in ((0, 1), (0, 2), (1, 2))):\n            raise ValueError(f"fold {number}: sample or text-group leakage")\n        if set.union(*sets) != set(range(len(paths))):\n            raise ValueError(f"fold {number}: saved partitions do not cover Training")\n        fold_indices.append((fit, stop))\n    # Verify the stored fold identities against the ORIGINAL deterministic run.\n    # We still use the stored indices below; a new split after warm-start is unsafe.\n    _, expected_cal, expected_folds, _ = base_pipeline.make_splits(\n        texts, targets, manifest["config"]["calibration_size"],\n        manifest["config"]["n_folds"], manifest["config"]["seed"],\n    )\n    if not np.array_equal(cal, expected_cal) or any(\n        not np.array_equal(actual, expected)\n        for pair, reference in zip(fold_indices, expected_folds)\n        for actual, expected in zip(pair, reference)\n    ):\n        raise ValueError("Saved fold identity differs from the original ver4 run; warm-start would leak.")\n    for number, name in enumerate(manifest["folds"], 1):\n        saved = torch.load(base_dir / name, map_location="cpu", weights_only=True, mmap=True)\n        if (saved.get("format_version") != 1\n                or saved.get("seed") != manifest["config"]["seed"] + number * 1009\n                or saved.get("architecture", {}).get("num_labels") != len(TARGET_SYMPTOMS)):\n            raise ValueError(f"Saved checkpoint fold identity mismatch: {name}")\n        del saved\n    with np.load(base_dir / "calibration_predictions.npz", allow_pickle=False) as cache:\n        names = cache["filenames"].tolist()\n        if len(set(names)) != len(names) or set(names) != set(splits["calibration"]):\n            raise ValueError("ver4 calibration cache filenames mismatch")\n        order = {name: i for i, name in enumerate(names)}\n        aligned = [order[paths[i].name] for i in cal]\n        raw_neural = np.array(cache["neural"][aligned], dtype=np.float64)\n        raw_sparse = np.array(cache["sparse"][aligned], dtype=np.float64)\n        if not np.array_equal(cache["targets"][aligned], targets[cal]):\n            raise ValueError("ver4 calibration cache targets mismatch")\n    for values in (raw_neural, raw_sparse):\n        if values.shape != (len(cal), len(TARGET_SYMPTOMS)) or not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):\n            raise ValueError("Invalid ver4 raw calibration probabilities")\n    fit, stop = fold_indices[init_fold - 1]\n    return manifest, paths, texts, targets, fit, stop, cal, group_ids, raw_neural, raw_sparse\n\n\ndef _run_signature(config, base_manifest, base_dir, init_checkpoint):\n    source_dir = Path(__file__).parent\n    code = hashlib.sha256()\n    own_code = [source_dir / name for name in RUNTIME_FILES if name.endswith(".py")]\n    for path in sorted([*own_code, *source_dir.joinpath("ver4_runtime").glob("*.py")]):\n        code.update(str(path.relative_to(source_dir)).encode())\n        code.update(path.read_bytes())\n    assets = [base_dir / name for name in [*base_manifest["folds"], base_manifest["sparse"], "mission3_ver4_model.ckpt"]]\n    assets += [path for path in (base_dir / base_manifest["tokenizer"]).rglob("*") if path.is_file()]\n    asset_hashes = {}\n    for path in sorted(assets):\n        name = str(path.relative_to(base_dir))\n        print("기본 모델 무결성 확인:", name, flush=True)\n        asset_hashes[name] = _sha256(path)\n    value = {\n        "config": config, "base_signature": base_manifest["signature"],\n        "initial_checkpoint_sha256": asset_hashes[str(init_checkpoint.relative_to(base_dir))],\n        "base_artifact_sha256": asset_hashes,\n        "calibration_cache_sha256": _sha256(base_dir / "calibration_predictions.npz"),\n        "splits_sha256": _sha256(base_dir / "splits.json"), "code_sha256": code.hexdigest(),\n        "versions": _versions(),\n    }\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest(), value\n\n\ndef _copy_file(source, target, expected_hash):\n    """An interrupted copy never appears under its final filename."""\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        if target.stat().st_size != source.stat().st_size or _sha256(target) != expected_hash:\n            raise ValueError(f"기존 묶음 파일 내용이 다릅니다: {target}")\n        return\n    temporary = target.with_name(target.name + ".tmp")\n    digest = hashlib.sha256()\n    with source.open("rb") as reader, temporary.open("wb") as writer:\n        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n            writer.write(chunk)\n            digest.update(chunk)\n    if digest.hexdigest() != expected_hash:\n        temporary.unlink()\n        raise ValueError(f"복사 중 기본 모델 파일이 변경되었습니다: {source}")\n    temporary.replace(target)\n\n\ndef _export_base(base_dir, output_dir, manifest, asset_hashes):\n    destination = output_dir / "base"\n    destination.mkdir(exist_ok=True)\n    # Copy to make the resulting ver6 folder portable without its old sibling.\n    for name in [*manifest["folds"], manifest["sparse"], "mission3_ver4_model.ckpt"]:\n        print("Bundle:", name, flush=True)\n        _copy_file(base_dir / name, destination / name, asset_hashes[name])\n    tokenizer_dir = base_dir / manifest["tokenizer"]\n    for path in tokenizer_dir.rglob("*"):\n        if path.is_file():\n            _copy_file(path, destination / manifest["tokenizer"] / path.relative_to(tokenizer_dir),\n                       asset_hashes[str(path.relative_to(base_dir))])\n    copied = base_pipeline.load_manifest(destination / "mission3_ver4_model.ckpt")\n    if copied["signature"] != manifest["signature"]:\n        raise ValueError("Portable base bundle signature mismatch")\n\n\ndef _export_code(output_dir):\n    source = Path(__file__).parent\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in RUNTIME_FILES:\n        if (source / name).is_file():\n            shutil.copyfile(source / name, submission / name)\n    for path in source.joinpath("ver4_runtime").iterdir():\n        if path.is_file():\n            destination = submission / "ver4_runtime" / path.name\n            destination.parent.mkdir(exist_ok=True)\n            shutil.copyfile(path, destination)\n\n\ndef train_pipeline(train_label_dir, base_dir, output_dir, config=None):\n    """Only TL_서울_구급 is accepted. Validation is deliberately not an argument."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    if not isinstance(config["bundle_base"], bool):\n        raise ValueError("bundle_base must be true or false")\n    if not np.isfinite(config["min_calibration_gain"]) or config["min_calibration_gain"] < 0:\n        raise ValueError("min_calibration_gain must be finite and nonnegative")\n    if type(config["min_calibration_improved_folds"]) is not int or config["min_calibration_improved_folds"] < 1:\n        raise ValueError("min_calibration_improved_folds must be a positive integer")\n    neural._settings(config)  # Reject invalid trials before hashing/extracting large inputs.\n    train_label_dir, base_dir, output_dir = (Path(x).expanduser().resolve() for x in\n                                          (train_label_dir, base_dir, output_dir))\n    if train_label_dir.name != "TL_서울_구급" or any("validation" in part.lower() for part in train_label_dir.parts):\n        raise ValueError("학습 입력은 서울 Training/TL_서울_구급 폴더여야 합니다.")\n    if output_dir == base_dir or output_dir.is_relative_to(base_dir) or base_dir.is_relative_to(output_dir):\n        raise ValueError("ver6 출력은 기존 ver4와 별도 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("ver6 학습에는 GPU 런타임을 선택하세요.")\n    (base_manifest, paths, texts, targets, fit, stop, cal, groups,\n     base_probs, sparse_probs) = _read_base(train_label_dir, base_dir, int(config["init_fold"]))\n    init_checkpoint = base_dir / base_manifest["folds"][int(config["init_fold"]) - 1]\n    signature, provenance = _run_signature(config, base_manifest, base_dir, init_checkpoint)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8")).get("signature") != signature:\n        raise ValueError("ver6 출력 폴더의 데이터/설정/초기 가중치/코드가 다릅니다. 새 OUTPUT_DIR를 지정하세요.")\n    if not run_path.exists() and any(output_dir.glob("*.ckpt")):\n        raise ValueError("run.json 없는 기존 checkpoint가 있습니다. 새 OUTPUT_DIR를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "provenance": provenance})\n    manifest_path = output_dir / "mission3_ver6_model.ckpt"\n    report_path = output_dir / "mission3_ver6_metrics.json"\n    if manifest_path.is_file() and report_path.is_file():\n        saved = load_manifest(manifest_path, base_dir=None if config["bundle_base"] else base_dir)\n        if saved["signature"] != signature:\n            raise ValueError("Final ver6 signature mismatch")\n        print("완료된 ver6 재사용: 추가 학습·보정 탐색을 건너뜁니다.", flush=True)\n        return manifest_path, json.loads(report_path.read_text(encoding="utf-8"))\n\n    write_json(output_dir / "splits.json", {\n        "fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop],\n        "calibration": [paths[i].name for i in cal], "init_fold": config["init_fold"],\n        "source": "Exact stored ver4 fold; no resplitting after warm-start",\n    })\n    print(f"ver4 재사용 | 새 모델 fit={len(fit):,}, stop={len(stop):,}, calibration={len(cal):,}", flush=True)\n    print(f"새 비교: 동결 특징 재사용 → head 후보 비교 → 선택 후보만 최대 {config[\'finetune_epochs\']} epoch "\n          f"미세조정 (상위 층={config[\'finetune_top_layers\']})", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(base_dir / base_manifest["tokenizer"], use_fast=True, local_files_only=True)\n    neural_config = {**config, "max_length": base_manifest["config"]["max_length"],\n                     "stride": base_manifest["config"]["stride"]}\n    documents = neural.tokenize_dialogues(texts, tokenizer, neural_config["max_length"], neural_config["stride"])\n    candidate_path, done_path = output_dir / "candidate_best.ckpt", output_dir / "candidate_done.json"\n    feature_report = None\n    if candidate_path.is_file() and done_path.is_file():\n        record = json.loads(done_path.read_text(encoding="utf-8"))\n        if record.get("signature") != signature:\n            raise ValueError("Candidate completion signature mismatch")\n        history = record["training"]\n        feature_report = record.get("feature_cache")\n        print("완료된 추가 모델 재사용", flush=True)\n    else:\n        # Only original fold fit/stop features are cached. Calibration targets\n        # and predictions are not passed to candidate training or head selection.\n        indices = np.concatenate([fit, stop])\n        feature_directory = config["feature_cache_dir"] or str(output_dir / "frozen_features")\n        store = _LazyFeatureStore(len(indices), feature_directory, lambda: features.build_feature_store(\n            [documents[i] for i in indices], tokenizer, init_checkpoint, feature_directory, neural_config,\n            checkpoint_sha256=provenance["initial_checkpoint_sha256"],\n        ))\n        try:\n            history = neural.train_model(\n                [documents[i] for i in fit], targets[fit], [documents[i] for i in stop], targets[stop],\n                tokenizer, neural_config, candidate_path, seed=int(config["seed"]) + 5003,\n                init_checkpoint_path=init_checkpoint, run_signature=signature, feature_store=store,\n                fit_indices=np.arange(len(fit)), stop_indices=np.arange(len(fit), len(indices)),\n                stop_groups=groups[stop],\n            )\n        finally:\n            store.close()\n        feature_report = store.report()\n        write_json(done_path, {"signature": signature, "training": history, "feature_cache": feature_report})\n    _release()\n    candidate_cache = output_dir / "candidate_calibration.npz"\n    candidate_digest = _sha256(candidate_path)\n    if candidate_cache.is_file():\n        with np.load(candidate_cache, allow_pickle=False) as cache:\n            if (cache["signature"].item() != signature or cache["filenames"].tolist() != [paths[i].name for i in cal]\n                    or cache["weights_sha256"].item() != candidate_digest):\n                raise ValueError("Candidate calibration cache mismatch")\n            candidate_probs = np.array(cache["probabilities"])\n    else:\n        candidate_probs = neural.predict_checkpoint(candidate_path, [documents[i] for i in cal], tokenizer, neural_config)\n        temporary = candidate_cache.with_name(candidate_cache.name + ".tmp")\n        with temporary.open("wb") as stream:\n            np.savez_compressed(stream, probabilities=candidate_probs,\n                                filenames=np.asarray([paths[i].name for i in cal]), signature=np.asarray(signature),\n                                weights_sha256=np.asarray(candidate_digest))\n        temporary.replace(candidate_cache)\n    _release()\n    # RAW ver4 predictions enter cross-fit; its full-calibration thresholds never do.\n    calibrator, report = calibration.calibrate_crossfit(\n        targets[cal], base_probs, sparse_probs, candidate_probs, groups[cal],\n        seed=int(base_manifest["config"]["seed"]) + 2,\n        n_splits=int(base_manifest["config"]["calibration_folds"]),\n        min_gain=float(config["min_calibration_gain"]),\n        min_improved_folds=config["min_calibration_improved_folds"],\n    )\n    report.update({\n        "signature": signature, "base_signature": base_manifest["signature"],\n        "training_samples": len(paths), "candidate_fit_samples": len(fit), "candidate_stop_samples": len(stop),\n        "candidate_training": history, "calibrator": calibrator, "config": config,\n        "feature_cache": feature_report,\n        "versions": _versions(), "validation_used_for_training_or_selection": False,\n        "new_models_trained": 1, "head_trials_compared": len(history.get("head_trials", [])),\n        "base_models_retrained": 0,\n    })\n    if config["bundle_base"]:\n        _export_base(base_dir, output_dir, base_manifest, provenance["base_artifact_sha256"])\n    else:\n        print("저장 공간 절약: 기존 ver4를 복사하지 않고 현재 폴더에서 재사용합니다.", flush=True)\n    _export_code(output_dir)\n    manifest = {\n        "format": "mission3_ver6", "version": 1, "signature": signature,\n        "target_symptoms": list(TARGET_SYMPTOMS), "portable": config["bundle_base"],\n        "base_manifest": ("base/mission3_ver4_model.ckpt" if config["bundle_base"] else\n                          str(base_dir / "mission3_ver4_model.ckpt")),\n        "base_signature": base_manifest["signature"], "candidate": "candidate_best.ckpt",\n        "config": neural_config, "calibrator": calibrator, "versions": _versions(),\n        "selected_name": report["selected_name"],\n    }\n    write_json(report_path, report)\n    _atomic_torch_save(manifest_path, manifest)\n    print(f"보정 교차평가: ver4={report[\'baseline_crossfit_macro_f1\']:.6f}, "\n          f"ver6 후보={report[\'candidate_crossfit_macro_f1\']:.6f}, 선택={report[\'selected_name\']}", flush=True)\n    print("이 점수는 Training 내부 선택용이며 공식 Validation 점수가 아닙니다.", flush=True)\n    return manifest_path, report\n\n\ndef load_manifest(checkpoint_path, base_dir=None):\n    """Validate a portable bundle or an explicit external-base experiment.\n\n    ``base_dir`` relocates an external ver4 dependency without rewriting the\n    saved manifest. Portable manifests always retain their internal base.\n    """\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver6" or manifest.get("version") != 1:\n        raise ValueError("mission3_ver6_model.ckpt를 지정하세요.")\n    if manifest.get("target_symptoms") != list(TARGET_SYMPTOMS):\n        raise ValueError("ver6 target label order mismatch")\n    root = checkpoint_path.parent\n    candidate = (root / manifest["candidate"]).resolve()\n    if not candidate.is_relative_to(root) or not candidate.is_file():\n        raise FileNotFoundError(f"ver6 결과 폴더 전체를 함께 보관하세요: {manifest[\'candidate\']}")\n    portable = manifest.get("portable", True)\n    if not isinstance(portable, bool):\n        raise ValueError("Invalid ver6 portable flag")\n    if portable:\n        if base_dir is not None:\n            raise ValueError("--base_dir is only supported for external-base ver6 experiments")\n        base_path = (root / manifest["base_manifest"]).resolve()\n        if not base_path.is_relative_to(root) or not base_path.is_file():\n            raise FileNotFoundError(f"ver6 결과 폴더 전체를 함께 보관하세요: {manifest[\'base_manifest\']}")\n    else:\n        base_path = Path(manifest["base_manifest"])\n        if not base_path.is_absolute():\n            raise ValueError("External ver4 manifest path must be absolute")\n        if base_dir is not None:\n            base_path = Path(base_dir).expanduser().resolve() / "mission3_ver4_model.ckpt"\n        if not base_path.is_file():\n            raise FileNotFoundError("기존 ver4 묶음을 다시 준비하고 --base_dir에 폴더를 지정하세요: "\n                                    f"{base_path}")\n        manifest["base_manifest"] = str(base_path.resolve())\n    base = base_pipeline.load_manifest(base_path)\n    if base.get("signature") != manifest.get("base_signature"):\n        raise ValueError("ver6 references a different ver4 base")\n    return manifest\n\n\ndef _predict_base_checkpoint(path, documents, tokenizer, config):\n    # Keep archived ver4 source bytes intact for its data/code fingerprint.\n    # Its old support check can count emulated BF16 on T4; use native precision\n    # during this synchronous inference call and restore the original helper.\n    original = base_neural._amp\n    base_neural._amp = features.amp_context\n    try:\n        return base_pipeline.predict_checkpoint(path, documents, tokenizer, config)\n    finally:\n        base_neural._amp = original\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None, cache_dir=None, base_dir=None):\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path, base_dir=base_dir)\n    root = checkpoint_path.parent\n    base_manifest = base_pipeline.load_manifest(root / manifest["base_manifest"])\n    _check_versions(base_manifest)\n    base_root = (root / manifest["base_manifest"]).parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = int(batch_size)\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(base_root / base_manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = neural.tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    cache = None\n    if cache_dir is not None:\n        execution = {"versions": {**_versions(), **{name: importlib.metadata.version(name)\n                     for name in ("scipy", "joblib")}}, "cuda_version": torch.version.cuda}\n        if config["device"] == "cuda":\n            execution.update(gpu=torch.cuda.get_device_name(),\n                             capability=list(torch.cuda.get_device_capability()),\n                             bf16=features.native_bf16_supported())\n        cache = PredictionCache(cache_dir, paths, texts, base_root / base_manifest["tokenizer"],\n                                TARGET_SYMPTOMS, execution)\n    source = Path(__file__).parent\n\n    def predict(kind, path, settings, sources, compute):\n        if cache is None:\n            return compute()\n        return cache.get(kind, path, settings, [source / name for name in sources], compute)\n\n    alpha = np.asarray(manifest["calibrator"]["neural_weights"])\n    beta = np.asarray(manifest["calibrator"]["candidate_weights"])\n    base_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    sparse_probs = np.zeros_like(base_probs)\n    candidate_probs = None\n    if np.any((1 - beta) * alpha > 0):\n        base_config = {**base_manifest["config"], "device": config["device"],\n                       "inference_batch_size": config["inference_batch_size"]}\n        # Keep ver4\'s original window batching for its numerical inference path.\n        for name in base_manifest["folds"]:\n            print("Predict base:", name, flush=True)\n            base_probs += predict("ver4-neural", base_root / name, base_config,\n                                  ["ver4_runtime/neural.py"],\n                                  lambda: _predict_base_checkpoint(base_root / name, documents, tokenizer, base_config))\n            _release()\n        base_probs /= len(base_manifest["folds"])\n    if np.any((1 - beta) * (1 - alpha) > 0):\n        sparse_probs = predict("ver4-sparse", base_root / base_manifest["sparse"], config,\n                               ["ver4_runtime/pipeline.py"],\n                               lambda: base_pipeline.predict_sparse(joblib.load(base_root / base_manifest["sparse"]), texts))\n    if np.any(beta > 0):\n        print("Predict: candidate_best.ckpt", flush=True)\n        candidate_probs = predict("ver6-neural", root / manifest["candidate"], config,\n                                  ["neural.py", "ver4_runtime/neural.py"],\n                                  lambda: neural.predict_checkpoint(root / manifest["candidate"], documents, tokenizer, config))\n        _release()\n    probabilities = calibration.apply_calibrator(base_probs, sparse_probs, candidate_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\nevaluate_csv = base_pipeline.evaluate_csv\n', 'train.py': '"""Train the complementary model and select calibration using Training only."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train_label_dir", required=True)\n    parser.add_argument("--base_dir", required=True)\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--config_json")\n    parser.add_argument("--auto_batch", action="store_true")\n    args = parser.parse_args()\n    config = json.loads(Path(args.config_json).read_text(encoding="utf-8")) if args.config_json else {}\n    if args.auto_batch:\n        run_path = Path(args.output_dir) / "run.json"\n        if run_path.is_file():\n            # Keep the original batch schedule. Incomplete neural training also\n            # verifies the GPU/precision identity before reusing epoch states.\n            saved = json.loads(run_path.read_text(encoding="utf-8"))["config"]\n            for key in ("batch_size", "window_batch_size", "gradient_accumulation_steps"):\n                config.setdefault(key, saved[key])\n            print("이전 실행의 batch 설정을 유지하여 재개합니다.", flush=True)\n        elif torch.cuda.is_available():\n            memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n            batch = 8 if memory_gb >= 35 else 4 if memory_gb >= 22 else 2\n            config.setdefault("batch_size", batch)\n            config.setdefault("window_batch_size", batch)\n            config.setdefault("gradient_accumulation_steps", 32 // config["batch_size"])\n    print("ver6 추가 학습 설정:", json.dumps(config, ensure_ascii=False), flush=True)\n    checkpoint, report = train_pipeline(args.train_label_dir, args.base_dir, args.output_dir, config)\n    print("Checkpoint:", checkpoint, flush=True)\n    print("선택:", report["selected_name"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""One-command Mission 3 inference; only transcript text is a model input."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import evaluate_csv, predict_pipeline\n    from .ver4_runtime.data import write_json\nelse:\n    from pipeline import evaluate_csv, predict_pipeline\n    from ver4_runtime.data import write_json\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", default="", help="Accepted for the competition interface; text task does not read audio.")\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--ckpt_path", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", choices=("cuda", "cpu"))\n    parser.add_argument("--batch_size", type=int)\n    parser.add_argument("--cache_dir", help="Optional persistent raw prediction cache shared across experiments.")\n    parser.add_argument("--base_dir", help="Relocated ver4 folder for an external-base ver6 experiment.")\n    parser.add_argument("--evaluate", action="store_true", help="Evaluate only after CSV is saved and all targets exist.")\n    args = parser.parse_args()\n    paths, _, _ = predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size,\n                                   cache_dir=args.cache_dir, base_dir=args.base_dir)\n    if args.evaluate:\n        target_count = sum("symptom" in json.loads(path.read_text(encoding="utf-8-sig")) for path in paths)\n        if target_count == len(paths):\n            report = evaluate_csv(args.label_dir, args.output)\n            write_json(Path(args.output).with_name("validation_metrics.json"), report)\n            print(f"Final evaluation Macro F1: {report[\'macro_f1\']:.6f}", flush=True)\n            for name, values in report["per_symptom"].items():\n                print(f"  {name}: {values[\'f1\']:.6f}", flush=True)\n        else:\n            print(f"정답 포함 JSON {target_count}/{len(paths)}: 예측 완료, 평가는 생략합니다.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'README.md': '# Mission 3 ver6\n\n기존 ver4 encoder를 재사용하며 **손실 함수, 직접적인 증상별 token attention, 작은 분류기의 정규화**를 비교한다.\n실행 파일은 `src/mission3/mission3_ver6.ipynb` 하나이며 필요한 Python 소스를 노트북에 포함한다.\n기존 **`mission3_ver5_inputs.zip`을 그대로 사용**한다. ver5 실행 결과나 새 데이터 ZIP은 필요 없다.\n\n## Colab에서 실행\n\n1. Google Drive 공유 문서함의 `mission3_ver5_inputs.zip`에서 **우클릭 → 정리 → 바로가기 추가 → 내 드라이브**를 선택한다.\n2. 내 드라이브 **바로 아래**에 같은 이름의 바로가기를 둔다. 파일을 다시 복사하거나 직접 압축을 풀 필요는 없다.\n3. `mission3_ver6.ipynb`를 Colab에서 열고 GPU 런타임을 선택한다.\n4. 위에서부터 모두 실행하고 바로가기를 만든 계정으로 Drive를 연결한다.\n\n첫 설정 셀의 기본 경로는 다음과 같다.\n\n```python\nINPUT_ZIP = "/content/drive/MyDrive/mission3_ver5_inputs.zip"\nBASE_DIR = None\nTRAIN_LABEL_DIR = None\nVALID_LABEL_DIR = None\nOUTPUT_DIR = None\nINFERENCE_CACHE_DIR = None\nFROZEN_FEATURE_CACHE_DIR = None\nRUN_VALIDATION = True\n```\n\n바로가기를 하위 폴더에 두었다면 `INPUT_ZIP`만 해당 절대 경로로 바꾼다.\nZIP에 Training·Validation JSON이 모두 있으므로 별도 공유 `DATA` 폴더는 연결하지 않는다.\nDrive를 연결한 후 바로가기를 추가해서 보이지 않으면 경로 준비 셀의\n`drive.mount("/content/drive", force_remount=True)`로 한 번 재연결한다.\n실행 중 계정을 바꾸지 말고 바로가기 대상 파일의 읽기 권한을 유지한다.\n\n## 입력과 새 결과의 위치\n\n| 용도 | 기본 경로 | 보관 방식 |\n|---|---|---|\n| 입력 ZIP 바로가기 | `/content/drive/MyDrive/mission3_ver5_inputs.zip` | Drive에 유지 |\n| 압축 해제한 기존 모델 | `/content/mission3_ver5_inputs/DCC_AIdeas/results/mission3/ver4` | Colab 로컬, 새 세션에서 재복원 |\n| Training·Validation JSON | `/content/mission3_ver5_inputs/DCC_AIdeas/DATA/대학부 데이터/` | ZIP에서 자동 연결 |\n| **새 ver6 결과** | **`/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver6`** | **Drive에 저장** |\n| 동결 encoder 특징 캐시 | `/content/mission3_ver6_features` | Colab 로컬, 같은 런타임에서 재사용 |\n| 원시 예측 캐시 | `/content/drive/MyDrive/DCC_AIdeas/results/mission3/prediction_cache` | Drive, 실험 간 조건 일치 시 재사용 |\n\n`results/mission3/ver4`라는 입력 폴더가 보이는 것은 정상이다. 이를 ver6으로 이름을 바꾸지 않는다.\nZIP은 약 6.86GB이며 ver4 필수 파일 13개, Training JSON 29,200개, Validation JSON 3,640개를 포함한다.\nWAV는 사용하지 않는다. ZIP 복원 시 manifest·CRC·SHA256을 검사하고, 같은 런타임에서 검증된 파일은 재사용한다.\n\nColab 세션이 삭제되면 압축 해제 파일과 특징 캐시는 사라진다. 새 세션에서는 ZIP을 다시 복원한다.\n특징은 **미완료 분류기 실험이 남았을 때만** 다시 계산한다. 분류기 비교가 끝나 encoder 미세조정만 재개하거나\n이미 완료된 후보·최종 결과를 재사용할 때는 동결 특징 추출을 생략한다.\n특징 캐시는 원래 Training fit·stop **26,320개**만 포함하며 별도 calibration **2,880개**는 제외한다.\n크기는 실제 token 수·hidden 크기로 추정하고 가용 공간을 확인한다. 해당 입력의 token별 encoder 특징을 저장하므로\n6개의 작은 분류기만 학습하는 시간 외에 최초 특징 추출·읽기·저장 비용이 존재한다.\n\n## ver5에서 바뀐 점\n\nver5는 기존 출력에 추가한 attention branch를 짧게 학습한 뒤, stop AP 하락으로 최적 epoch가 0에 머물렀다.\n이번에는 ver4를 공통 출발점으로 삼아 아래 실험을 수행한다. 학습 입력의 label에 따라 텍스트 전처리를 바꾸지 않는다.\n\n| 실험 | 분류기 | 손실·정규화 |\n|---|---|---|\n| A | 기존 ver4 global/window 분류기 | Asymmetric Loss |\n| B | 기존 ver4 global/window 분류기 | Distribution-Balanced Loss |\n| C | 기존 ver4 global/window 분류기 | ASL + soft Macro F1 |\n| D | 직접적인 label-wise token attention | Weighted BCE |\n| E | D | Weighted BCE + 분류기의 Bernoulli R-Drop |\n| F | E + 작은 label dependency 모듈 | fit에서 구한 증상 관계 활용 |\n\nA–F는 **한 번 추출한 동결 encoder 특징**을 공유한다. 각 실험별로 전체 encoder 체크포인트를 복제하지 않는다.\nA–C는 분류기에 필요한 CLS·평균 요약을 메모리에 재사용하여 매 epoch마다 전체 token 특징을 다시 읽지 않는다.\nD–F는 증상별 attention을 위해 token별 특징을 사용한다.\nD–F에서는 각 증상의 query가 token 표현을 선택하고 window를 모아 분류한다.\n최종 출력이 해당 표현에 직접 의존하며, ver5처럼 기존 logit에 0으로 시작하는 branch만 더하는 방식이 아니다.\nR-Drop의 두 dropout 경로는 작은 분류기에 적용한다. 최종 encoder 미세조정에서는 기본적으로 R-Drop을 끈다.\nsoft Macro F1 보조항도 작은 분류기 단계에만 사용하며 encoder 미세조정의 작은 microbatch에는 적용하지 않는다.\n\n선택된 분류기 하나만 encoder 상위 2개 층과 최대 1 epoch 미세조정한다.\n이 단계에서 stop F1이 떨어지면 미세조정 전 가중치를 유지한다. encoder를 크게 바꾸거나 더 큰 모델을 내려받지 않는다.\n**ver6의 최종 `best_epoch=0`은 선택된 분류기를 유지하고 encoder 미세조정만 채택하지 않았다는 뜻이다.**\n분류기 학습이 최종 가중치에 반영됐는지는 `head_trials[].best_epoch`에서 따로 확인한다.\n이미 학습한 epoch가 선택될 수도 있고, 해당 분류기의 초기 epoch 0이 유지될 수도 있다.\n원 논문의 모델·실험 전체를 그대로 재현하는 것은 아니다. 출처·구현 대응·제외 범위는 [RESEARCH.md](RESEARCH.md)를 확인한다.\n\n## 데이터 분할과 선택 기준\n\n- 원래 ver4 fold 1의 **fit 21,032개**만 학습에 사용한다. class 빈도와 label dependency 통계도 fit에서만 계산한다.\n- 원래 **stop 5,288개**의 대화 그룹을 분리한 3-fold threshold 교차평가로 epoch·A–F 후보를 비교한다.\n  각 평가 fold의 threshold는 나머지 stop fold에서 맞춘다. Macro F1을 우선하고 AP는 보조로 기록한다.\n- 별도 **Training calibration 2,880개**의 5-fold 교차평가로 기존 앙상블·새 후보의 혼합 비율과 임계값을 비교한다.\n- 최종 후보는 기존 calibration 교차평가 Macro F1보다 **0.002 이상** 좋고, **5개 fold 중 최소 3개**에서\n  F1이 엄격히 개선되어야 선택한다. 기준을 충족하지 못하거나 동점이면 기존 ver4 보정을 유지한다.\n- **공식 Validation 3,640개 전체**는 최종 선택 후 예측·평가에만 사용한다. 그 정답으로 재학습·재보정하지 않는다.\n\n이 비교는 고정된 ver4 fold 1과 내부 threshold 교차평가를 사용한다.\n**5개 encoder를 각각 새로 학습한 5-fold 모델 OOF 실험이 아니다.** 같은 stop에 여러 후보·epoch를 비교하므로\nstop 점수에는 선택 편향이 남는다. calibration도 반복 연구에 계속 활용하면 선택 편향이 생길 수 있다.\n새 구성을 고르는 최소 개선 기준은 작은 변화의 과대해석을 줄이는 장치이며, Validation 또는 Test 개선을 보장하지 않는다.\n\n이전 전체 Validation 기록은 ver4 **0.6566489480388843**, ver5 **0.6567177583831921**이다.\nver6의 새 F1은 실제 실행 전에는 알 수 없다. notebook은 마지막에 이 두 기록과 비교하며 차이를 선택 과정에 되먹이지 않는다.\n\n## 기본 학습 설정\n\n| 설정 | 기본값 |\n|---|---|\n| `profiles` | `A, B, C, D, E, F` |\n| `head_epochs`, `head_patience` | 4, 2 |\n| `head_batch_size`, `head_learning_rate` | 32, `3e-4` |\n| `stop_crossfit_splits` | 3 |\n| `finetune_epochs`, `finetune_top_layers` | 1, 2 |\n| `learning_rate`, `finetune_head_learning_rate` | `2e-6`, `5e-5` |\n| `finetune_rdrop` | `False` |\n| `min_calibration_gain`, `min_calibration_improved_folds` | 0.002, 3 |\n| notebook `bundle_base` | `False` |\n\nCLI의 `--auto_batch`는 GPU 메모리에 따라 encoder batch/window 크기를 선택한다.\n같은 실험을 재개할 때는 저장된 batch 설정을 유지한다. **미완료 학습 재개에는 같은 GPU 종류·AMP 정밀도와\n관련 런타임 패키지 버전이 필요하다.** 다른 실행 조건의 stop 점수가 섞이지 않도록 중간 저장 파일에서 검사한다.\n최종 결과까지 완료된 실행은 다른 GPU에서도 재사용할 수 있다. 이때 기존 원시 예측 캐시의 실행 조건이 다르면 다시 추론한다.\n학습 설정·코드·데이터를 바꾼다면\n새 `OUTPUT_DIR`을 지정한다. 동일한 런타임에서는 호환되는 특징 캐시를 재사용할 수 있다.\n총 CU는 아직 실측하지 않았으며, 높은 GPU 사양이 항상 낮은 총 CU를 의미하지는 않는다.\n\n## 저장 결과와 추론\n\n기본 notebook은 기존 5개 모델을 결과 폴더에 다시 복사하지 않는 `bundle_base=False`로 실행한다.\n**결과 폴더와 원래 입력 ZIP을 함께 보관해야 한다.** 별도 CLI에서 `bundle_base=True`를 설정하면\n기존 모델까지 포함한 출력 묶음을 만들 수 있지만 저장 용량이 증가한다.\n\n- `mission3_ver6_model.ckpt`: 최종 구성·보정값을 저장한 manifest.\n- `candidate_best.ckpt`: 선택된 후보의 encoder와 분류기 가중치.\n- `mission3_ver6_metrics.json`: A–F 학습 이력·선택, 미세조정, calibration 교차평가와 fallback 이유.\n- `predictions/<실행시각>/mission3.csv`: 제출 형식 예측.\n- 같은 예측 폴더의 `validation_metrics.json`: 정답이 있을 때 전체·증상별 precision, recall, F1, support.\n- `notebook_runtime_<실행시각>.json`: 입력 준비, 학습·보정, 최종 추론·평가의 벽시계 시간. 패키지 설치 시간과 CU는 별도.\n- `submission/`: 다른 환경에서 실행할 추론 소스·패키지 목록·문서.\n\n```bash\npython submission/inference.py --audio_dir "WAV 폴더" --label_dir "JSON 폴더" \\\n  --ckpt_path mission3_ver6_model.ckpt --base_dir "ZIP에서 복원한 ver4 폴더" \\\n  --output mission3.csv\n```\n\n`--audio_dir`는 대회 실행 인터페이스와의 호환용이며 음성은 읽지 않는다.\n`--cache_dir`로 공유 예측 캐시를 지정할 수 있다. 정답과 보정값은 원시 예측 캐시에 저장하지 않으며,\n입력 text·가중치·tokenizer·추론 코드·관련 실행 조건이 같은 경우에만 재사용한다.\n`--evaluate`는 CSV 저장 이후 정답이 있는 입력을 평가하며 모델을 변경하지 않는다.\n\n## 노트북 다시 생성\n\n```bash\npython src/mission3/mission3_ver6/build_notebook.py\n```\n\n노트북은 현재 runtime 소스를 포함한다. runtime 수정 후 재생성해야 Colab에 올린 단일 파일에 변경 내용이 반영된다.\n로컬의 작은 CPU 검증은 수치·재개·캐시·추론 구현을 확인하는 용도이고 실제 전체 데이터의 F1·GPU 시간 검증을 대신하지 않는다.\n', 'RESEARCH.md': '# Mission 3 ver6: 논문 근거와 적용 범위\n\n검토일: 2026-09-15. 사용자가 제공한 `## 지금 상태에서 어디를 건드려야 하는지부터.txt`를 출발점으로 삼고, 아래 원문 PDF의 방법 절과 저자 코드를 확인했다. 논문의 다른 데이터셋 성능을 Mission 3의 예상 상승폭으로 환산하지 않는다. **ver6의 실제 F1과 총 Colab 컴퓨팅 단위는 실행 결과로 확인해야 한다.**\n\n### 대회 원문과 대조한 조건\n\n[예선 출제문제](../../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–11쪽을 직접 확인했다. 서울 Training만 학습에 쓰고, 입력은 전사 text만 사용한다. 정답에 따라 입력 전처리를 달리하거나 공식 Validation을 학습에 사용하지 않는다. 지정된 9개 외의 symptom만 제외하며 **샘플 자체는 유지**하므로 9개 target이 모두 0인 통화도 포함한다. 최종 CSV는 label 파일명과 0–9개 symptom 문자열을 출력한다. 소스의 `dialogue_text`와 추론의 `include_targets=False` 경로가 이 분리를 유지한다.\n\n## 1. 논문 원문\n\nPDF 6개를 로컬 저장소의 `mission3_ver6/papers/`에 보관했다. 페이지는 PDF 뷰어의 1부터 시작하는 페이지 번호다. 논문은 개발 근거 자료이며 노트북 실행 입력이나 학습 데이터가 아니다. Colab에는 기존 입력 ZIP과 새 노트북만 있으면 된다. 내보낸 실행 코드에는 PDF를 포함하지 않으며, 그 환경에서는 표의 원문 URL로 확인할 수 있다.\n\n| 원문 | 실제 확인한 범위 | 보관 파일 |\n|---|---|---|\n| Huang et al., **PLM-ICD**, ClinicalNLP 2022 | PDF 4–6쪽 §3–5, 8쪽 §6: segment pooling, label attention, 한계·ablation | [원문](https://aclanthology.org/2022.clinicalnlp-1.2.pdf) · [로컬](papers/plm_icd_2022.pdf) |\n| Vu et al., **LAAT**, IJCAI 2020 | PDF 2–4쪽 §3–4: attention 식 (4)–(6), label classifier, JointLAAT와 학습 설정 | [원문](https://arxiv.org/pdf/2007.06351) · [로컬](papers/laat_2020.pdf) |\n| Ben-Baruch, Ridnik et al., **ASL**, ICCV 2021 / arXiv v4 | PDF 2–4쪽 §2: 식 (2)–(7), focusing·clipping·gradient 분석 | [원문](https://arxiv.org/pdf/2009.14119) · [로컬](papers/asl_2021.pdf) |\n| Wu et al., **Distribution-Balanced Loss**, ECCV 2020 | PDF 4–7쪽 §3, 저자 `ResampleLoss`와 COCO 설정: rebalancing, bias, negative scaling | [원문](https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123490154.pdf) · [로컬](papers/db_loss_2020.pdf) |\n| Bénédict et al., **sigmoidF1**, arXiv v3 (2022; 최초 공개 2021) | PDF 7–9쪽 §4.3–5, 25쪽 hyperparameters: soft confusion counts, batch 단위 Macro-F1 | [원문](https://arxiv.org/pdf/2108.10566) · [로컬](papers/sigmoidf1.pdf) |\n| Liang et al., **R-Drop**, NeurIPS 2021 | PDF 2–4쪽 §2: 독립 dropout 출력 두 개, symmetric KL, 학습 비용 | [원문](https://proceedings.neurips.cc/paper_files/paper/2021/file/5a66b9200f29ac3fa0ae244cc2a51b39-Paper.pdf) · [로컬](papers/r_drop_2021.pdf) |\n\n### TXT의 2026 IEEE 논문 확인\n\n*Mitigating Multi-label Imbalance in ICD Coding: A Comprehensive Evaluation of Loss Functions and Threshold Optimization*의 제목·저자·IEEE Access 게재 예정 표기를 [저자 연구실의 공식 목록](https://mix.seoultech.ac.kr/publications)에서 확인했다. 목록은 [IEEE 문서 11589268](https://ieeexplore.ieee.org/document/11589268)을 연결한다. IEEE 원문 본문은 이번 환경에서 가져오지 못했으므로, **해당 논문의 수치나 비교 우열을 ver6 설계 근거로 사용하지 않았다.** 위 여섯 원문으로 구현에 필요한 방법을 확인했다.\n\n## 2. 직접 분류하는 label attention\n\nPLM-ICD는 문서의 각 segment를 encoder에 통과시킨 뒤 **token hidden state를 연결**하고, LAAT의 증상별 attention을 적용한다. 아래 식에서 `H`의 열은 token, `A`의 행은 label이다.\n\n```text\nH = concat(encoder(window_1), ..., encoder(window_W))\nA = softmax_token(U tanh(V H))\nR = H A^T\nz_k = w_k^T R_k + b_k\np_k = sigmoid(z_k)\n```\n\nver6의 D/E/F 후보는 이 표현으로 **최종 logit을 직접 계산**한다. ver5의 작은 residual 분기와 구조가 다르다. 각 증상 query가 같은 전사 전체를 보며, 정답 label에 따라 입력 token을 바꾸지 않는다. [PLM-ICD §4.2–4.3](https://aclanthology.org/2022.clinicalnlp-1.2.pdf) · [LAAT §3.1](https://arxiv.org/pdf/2007.06351)\n\nMission 3 적용 차이는 기존 한국어 encoder와 512-token window/64-token overlap 재사용, padding·special token을 attention 후보에서 제외하는 것이다. overlap에 해당하는 token 표현은 서로 다른 문맥의 표현으로 포함된다. 원문의 영어 임상 사전학습, ICD 계층, 추가 의료 corpus는 도입하지 않는다. Attention 가중치 자체를 원인 설명이나 의학적 근거로 간주하지 않는다.\n\n## 3. 비용을 제한한 여섯 가지 비교\n\n기존 ver4의 fold 1 encoder를 고정하고 입력별 token 표현을 한 번 추출한다. A–F는 **같은 캐시와 같은 fit/stop 분할**을 쓰는 작은 분류기 실험이다. 모델 여섯 개의 encoder를 각각 미세조정하는 실험이 아니다.\n\n| 후보 | 분류기와 손실 | 확인하려는 변화 |\n|---|---|---|\n| A | 기존 window 분류기 + ASL | 구조를 유지한 손실 변경 |\n| B | 기존 window 분류기 + DB-Loss | 빈도·공발생을 반영한 손실 변경 |\n| C | 기존 window 분류기 + ASL + `0.2 × soft Macro-F1` | 공식 지표에 가까운 보조 학습 목표 |\n| D | 직접 label-token attention + weighted BCE | 증상별 token 표현 자체의 효과 |\n| E | D + 분류기 단계의 Bernoulli R-Drop | 작은 추가 비용의 출력 일관성 |\n| F | E + Training fit으로 추정한 9-label dependency | 공발생 정보의 추가 효과 |\n\n분류기 비교는 후보당 최대 4 epoch, 실제 batch 32, 개선 없는 2 epoch 후 중단이 기본이다. 선택된 분류기 하나만 encoder 상위 2개 층을 최대 1 epoch 미세조정한다. 이때 기본값은 R-Drop과 soft-F1 항을 끈다. 전자는 encoder를 두 번 계산하는 비용, 후자는 작은 GPU microbatch의 F1 통계 불안정을 피하기 위한 절충이다. 캐시 표현을 만든 encoder와 미세조정된 encoder는 다르므로 **미세조정 모델의 추론에 이전 고정 표현을 사용하면 안 된다.** 실제 추론과 미세조정 전후 비교는 선택된 encoder를 직접 실행한다.\n\n### ASL: 음성·양성의 gradient 기여를 다르게 조절\n\n```text\np = sigmoid(z)\np_m = max(p - margin, 0)\nloss = -y (1-p)^gamma_pos log(p)\n       -(1-y) p_m^gamma_neg log(1-p_m)\n```\n\n기본값은 `gamma_pos=0, gamma_neg=2, margin=0`. ASL 후보에는 기존 BCE의 `pos_weight`를 중복 적용하지 않는다. 원문의 clipping은 매우 쉬운 음성뿐 아니라 일부 매우 어려운 음성의 gradient도 줄인다. 따라서 TXT의 “ASL이면 false positive가 줄어든다”는 설명은 검증할 가설이다. 현재 기본값은 clipping을 끄고, 증상별 precision/recall/F1으로 실제 영향을 확인한다. [ASL §2](https://arxiv.org/pdf/2009.14119)\n\n### DB-Loss: 저자 구현의 부호·정규화 기준\n\nfit에서 얻은 클래스 빈도 `n_k`, fit 크기 `N`을 사용한다. 재가중치는 `r_k = (1/n_k) / sum_j(y_j/n_j)`에 `0.1 + sigmoid(10(r_k-0.2))`를 적용한다. 정답이 모두 0인 행은 분모 0을 나누지 않고, 이 식의 극한인 `1.1`을 사용한다.\n\n저자의 COCO 설정을 출발점으로 `negative_scale=2`, `init_bias=0.05`, focal `gamma=2`, multiplier `2`를 쓴다. bias는 `0.05/2 × log(p_k/(1-p_k))`를 logit에 더한다. 음성 logit에는 2를 곱하고 음성 loss weight에는 1/2를 곱한 뒤 BCE와 focal 항을 계산한다. 빈도 0/1은 수치적으로 보호한다. 부호와 bias의 scale 나눗셈은 [저자 구현](https://github.com/wutong16/DistributionBalancedLoss/blob/master/mllt/models/losses/resample_loss.py)과 [COCO 설정](https://github.com/wutong16/DistributionBalancedLoss/blob/master/configs/coco/LT_resnet50_pfc_DB.py)을 기준으로 확인했다.\n\n원문의 class-aware 재표집까지 재현하지 않고, 같은 입력 순서를 유지하는 **손실만 적용한 비교**다. label에 따른 loss weight는 정답을 쓰는 지도학습 목적함수이며, label에 따른 입력 전처리와 구분한다. [DB-Loss §3](https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123490154.pdf)\n\n### soft Macro-F1: sigmoidF1에서 단순화한 보조항\n\n실제 minibatch에 대해 `s=sigmoid(z)`, `TP_k=sum(s_k*y_k)`, `FP_k=sum(s_k*(1-y_k))`, `FN_k=sum((1-s_k)*y_k)`를 계산하고 `1-mean_k(2TP_k/(2TP_k+FP_k+FN_k+eps))`를 사용한다. 원문의 sigmoid slope/offset은 `1/0`으로 고정한 형태다. 해당 batch에 양성이 없는 label은 이 보조항의 평균에서 제외하며, ASL은 그 음성도 계속 학습한다.\n\n이는 **ASL과 결합한 soft Macro-F1 변형**이며 논문 전체 재현이 아니다. sigmoidF1 원문도 충분한 batch 통계가 필요하다고 설명한다. Gradient accumulation으로 여러 작은 batch의 loss를 합쳐도, 큰 batch에서 F1의 분자·분모를 한 번 합산한 값과 같지 않다. [sigmoidF1 §4.3–4.4](https://arxiv.org/pdf/2108.10566)\n\n### R-Drop과 label dependency의 적용 한계\n\nR-Drop 원문은 같은 입력에 독립적인 dropout을 적용한 두 모델 출력을 일치시킨다. ver6의 E/F는 **고정 encoder 표현 위의 분류기만 두 번 계산**하는 변형이다. 두 개의 지도학습 loss 평균에 각 label의 Bernoulli 분포 `[p, 1-p]` 사이 symmetric KL을 더한다. 9개 label은 동시에 참일 수 있으므로 label 축 softmax-KL을 쓰지 않는다. encoder dropout까지 정규화하는 원문 방식과 같은 효과라고 주장하지 않는다. [R-Drop §2](https://proceedings.neurips.cc/paper_files/paper/2021/file/5a66b9200f29ac3fa0ae244cc2a51b39-Paper.pdf)\n\nF의 작은 dependency 분류기는 fit label의 공발생을 초기 정보로 사용한다. 정답 label을 추론 입력으로 넣지 않는다. 이는 Mission 3용 실험이며 TXT의 *Orderless Recurrent Models*나 LAAT의 ICD 계층 학습을 재현한 것으로 표기하지 않는다. 별도 Longformer·SMART·외부 지식·LLM 데이터 증강은 기본 실행에 포함하지 않는다.\n\n## 4. 모델 선택과 성능 해석\n\n- 기존 fold 1의 **fit 21,032 / stop 5,288**을 유지한다. 새 분류기의 손실·빈도·공발생 학습은 fit만 사용한다.\n- stop 안에서 3분할로 임계값을 교차 추정한 Macro-F1을 사용해 후보와 epoch를 비교한다. 임계값을 학습한 동일 행에서 바로 F1을 채점하는 낙관을 줄인다.\n- 별도 Training calibration 2,880개에서 기존 ensemble과 신규 후보 혼합을 평가한다. 기본 채택 조건은 기존 baseline 대비 교차평가 Macro-F1 **0.002 이상** 상승과 5분할 중 **3개 이상** 상승이다. 이는 작은 우연한 변동을 거르는 운영 규칙으로, 통계적 유의성 보장이 아니다.\n- 이 과정은 **전체 Training으로 신경망 5개를 새로 학습한 완전한 5-fold OOF가 아니다.** stop은 반복적인 후보 선택에 쓰이고, calibration도 이전 버전부터 재사용되어 선택 편향이 남는다.\n- 공식 Validation 3,640개는 최종 선택 후 보고용으로만 사용한다. Validation F1을 보고 자동 재학습·threshold 조정·후보 선택을 하지 않는다.\n- 모든 후보에 같은 text normalization/tokenization/window 규칙을 적용한다. 표본별 정답을 보고 문장이나 token을 추가·삭제하지 않는다.\n\n현재 baseline은 강하고 ver5의 개선폭은 작았다. ver6는 구조와 손실을 실제로 비교할 수 있게 만드는 변경이다. 새 후보가 조건을 충족하지 못하면 기존 baseline을 유지하며, 큰 F1 상승을 만들어 낸 것으로 기록하지 않는다.\n\n## 5. 원문 파일 검증\n\n공개 원문 URL에서 받은 PDF 6개를 로컬 PDF parser로 열어 페이지 수와 본문을 확인했다. 아래 SHA-256은 보관 파일을 식별한다.\n\n| 파일 | PDF 페이지 수 | SHA-256 |\n|---|---:|---|\n| `plm_icd_2022.pdf` | 11 | `6b65a85525dc1695fbe000afac28ef8c01d87353f5f3654a7718cefffa943577` |\n| `laat_2020.pdf` | 7 | `efe4c78d61c7a12562ed41bfae0b32b44f873d449babee4b264c84bee87e8b9f` |\n| `asl_2021.pdf` | 13 | `730d9af30d43febe7ebd7c5ae5c23ed46d35e8c76f1a6b711249346dbcd80c73` |\n| `sigmoidf1.pdf` | 28 | `b241be14ef5712aadcf024d224130383c603c91ce7acf9eb7b81e010e5567446` |\n| `db_loss_2020.pdf` | 17 | `36640358c039bc0a118434c6ccc31d2dbfba3d6a2c4e48b64269934a3db36439` |\n| `r_drop_2021.pdf` | 16 | `4f3dcee5dcc4951586f1b599076e18abb24dca7ae669e62b2b8bafb560d5a331` |\n', 'ver4_runtime/README.md': '# Mission 3 ver4 실행 안내\n\n실행 파일은 [src/mission3_ver4.ipynb](../mission3_ver4.ipynb)입니다. 이 노트북 하나만 Colab에 올리면 필요한 모델·학습·추론 코드를 생성합니다. 이 폴더의 `.py` 파일은 같은 기능을 로컬이나 제출 환경에서 실행하기 위한 코드입니다.\n\n## 기존 결과와 개선 근거\n\n| 실행 버전 | 실제 Validation Macro F1 | 근거 |\n|---|---:|---|\n| `mission3_all_in_one_ver2` → results/mission3/ver1 | 0.600123 | `results/mission3/ver1/mission3_metrics.json` |\n| `mission3_ver3` → results/mission3/ver3 | 0.626015 | `results/mission3/ver3/mission3_ver3_metrics.json` |\n| ver4 | 아직 미측정 | 전체 GPU 학습 후 평가 필요 |\n\nver3의 Training 내부 F1은 3 epoch에서 0.644361로 최고였고, 15 epoch에는 0.601536으로 하락했습니다. 학습 손실은 계속 감소해 과적합 징후가 있습니다. ver4는 별도 stopping 집합의 Macro AP로 최적 epoch를 선택하고 조기 종료합니다. AP는 임계값 없이 예측 순위를 평가하며, 최종 대회 지표는 그대로 **9개 증상의 Macro F1**입니다.\n\nTraining 29,200개를 저장된 ver3 tokenizer로 직접 측정했을 때 384토큰을 넘는 대화는 3,292개(11.27%), 512토큰을 넘는 대화는 788개(2.70%)였습니다. 특수 토큰을 포함한 최대 길이는 873토큰입니다. ver4는 긴 대화의 뒷부분도 처리하도록 설계했습니다.\n\n**현재 작성 환경에는 GPU가 없어 ver4 전체 학습과 Validation 성능은 확인하지 못했습니다.** 작은 CPU 모델을 이용한 실행 검증은 코드 경로 확인용입니다. 기존 버전보다 성능이 높다는 판단은 전체 학습·추론이 끝난 뒤 실제 점수로 해야 합니다.\n\n## 모델과 데이터 분리\n\n- `klue/roberta-large`를 5개 fold에서 각각 학습하고 예측 확률을 평균냅니다.\n- 한 window는 최대 512토큰이며 인접 window는 64토큰이 겹칩니다. 마지막 window까지 유지하고, 증상별 attention으로 대화 전체를 모은 뒤 **통화당 한 번** BCE 손실을 계산합니다.\n- 문자 2–5그램과 단어 1–2그램 TF-IDF + 증상별 Logistic Regression을 함께 학습합니다.\n- 증상마다 신경망/TF-IDF의 혼합 비율과 판정 임계값을 Training 내부에서 결정합니다. 어떤 통화도 최소 1개 증상을 출력하도록 강제하지 않습니다.\n\n서울 Training을 약 90% development와 10% calibration으로 먼저 나눕니다. Development 내부 5-fold는 원래 Training 기준으로 각 fold마다 약 72% fit, 18% stopping입니다. 동일한 정규화 대화는 같은 그룹에 두어 분할을 넘지 않게 합니다.\n\nCalibration 10%는 신경망 학습, TF-IDF 학습, epoch 선택에 전혀 사용하지 않습니다. TF-IDF는 development 90%에만 fit합니다. 완성된 5개 신경망의 평균 확률과 TF-IDF 확률을 calibration에 적용한 뒤, 그 안에서 다시 5-fold 교차 보정하여 혼합 비율·임계값 선택의 일반화 점수를 계산합니다. 배포용 보정값은 calibration 전체로 결정합니다. **보정 후 기본 모델을 다시 학습하지 않고 같은 모델들을 그대로 추론에 사용합니다.** 공식 Validation은 이 과정에 들어가지 않습니다.\n\n주요 기본값은 `pipeline.py`의 `DEFAULT_CONFIG`에 있습니다.\n\n| 설정 | 기본값 |\n|---|---|\n| 모델 / fold | `klue/roberta-large` / 5 |\n| 최대 epoch / 조기 종료 patience | 6 / 2 |\n| 통화 batch / gradient accumulation | 2 / 16 |\n| encoder / head learning rate | `1.5e-5` / `1e-4` |\n| 문자 / 단어 최대 특징 수 | 120,000 / 40,000 |\n\n## Colab에서 실행\n\n1. [mission3_ver4.ipynb](../mission3_ver4.ipynb)를 Colab에 업로드합니다.\n2. GPU 런타임을 선택합니다. 기본 large 모델의 5-fold 학습에는 **A100 또는 L4를 권장**합니다. 사용 가능한 장치에 따라 시간과 메모리 사용량은 달라집니다.\n3. 제공 데이터의 압축을 풀고 `TL_서울_구급`, `VL_서울_구급` 폴더 이름을 유지합니다. Mission 3은 JSON만 읽습니다.\n4. 위에서부터 모든 셀을 실행하고 Google Drive를 연결합니다. 노트북은 `/content`를 먼저 탐색하고 Drive의 알려진 데이터 경로를 확인합니다. 자동 탐색에 실패하면 노트북의 데이터 경로 설정에 실제 폴더를 지정합니다.\n5. 학습·보정이 끝난 뒤 Validation 추론과 평가 셀을 실행합니다. 실행 로그가 남은 노트북도 저장합니다.\n\n기본 결과 폴더는 Colab에서 `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4`, 로컬에서 프로젝트의 `results/mission3/ver4`입니다. 신경망 가중치 5개를 저장하므로 Drive 여유 공간도 확인합니다. 패키지와 최초 공개 pretrained 모델 다운로드에는 인터넷 연결이 필요하며, 학습 완료 후에는 저장된 구성으로 오프라인 추론할 수 있습니다.\n\n## 로컬 또는 제출 환경에서 실행\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트 기준입니다. 전체 학습 명령은 CUDA GPU를 요구합니다.\n\n```bash\npython -m pip install -r src/mission3_ver4/requirements.txt\n\npython src/mission3_ver4/train.py \\\n  --label_dir "DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급" \\\n  --output_dir results/mission3/ver4\n\npython src/mission3_ver4/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver4/mission3_ver4_model.ckpt \\\n  --output results/mission3/ver4/mission3.csv\n\npython src/mission3_ver4/evaluate.py \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --prediction results/mission3/ver4/mission3.csv \\\n  --output results/mission3/ver4/validation_metrics.json\n```\n\n`--audio_dir`는 주최 측 인터페이스를 맞추기 위해 받으며 음성 파일을 읽지 않습니다. 추론은 `--device cpu`도 지원하지만 large 모델 5개를 처리하므로 GPU보다 느릴 수 있습니다. 학습 설정은 `train.py --config 설정.json`으로 변경할 수 있습니다. 설정을 바꿀 때는 새 `--output_dir`를 사용합니다.\n\n## 결과 읽기와 파일 보관\n\n| 파일 | 의미 |\n|---|---|\n| `mission3_ver4_metrics.json` | Training 내부 보정 점수, 증상별 교차 보정 F1, 최종 보정값 |\n| `mission3_ver4_history.json` | 각 fold의 epoch별 학습·stopping 기록 |\n| `validation_metrics.json` | 위 평가 명령으로 만든 공식 Validation Macro F1와 증상별 점수 |\n| `mission3.csv` | `label file name,symptom` 형식의 예측 결과 |\n| `run.json`, `splits.json` | 실행 설정·데이터/코드 식별값과 분할 파일 목록 |\n| `submission/` | 실행 당시 `.py` 파일, requirements와 실행 안내 사본 |\n\n`calibration_crossfit_macro_f1`는 각 보정 평가 샘플의 정답을 보정값 선택에서 제외하여 계산한 **Training 내부 점수**입니다. `calibration_fit_macro_f1`는 같은 calibration 데이터에서 보정값을 정하고 다시 평가한 점수라 낙관적일 수 있습니다. 둘을 Validation 점수로 보고하면 안 됩니다. 공식 Validation 평가를 모델 선택이나 임계값 재조정에 반복 사용하지 않습니다.\n\n**`mission3_ver4_model.ckpt`는 구성 목록과 보정값을 담은 manifest입니다. 이 파일 하나만 옮기면 추론할 수 없습니다.** 결과 디렉터리 전체를 함께 보관하거나 최소한 다음 구성을 유지합니다.\n\n```text\nver4/\n├─ mission3_ver4_model.ckpt\n├─ fold_1.ckpt ... fold_5.ckpt\n├─ tokenizer/\n├─ sparse.joblib\n└─ submission/\n```\n\nCSV의 symptom 값은 기존 결과와 같은 문자열 목록 표현입니다. 예를 들어 `[]`, `[\'구토\', \'복통\']`처럼 저장하며 CSV 인용 처리는 코드가 담당합니다. 주최 측 sample submission에서 더 구체적인 형식을 제공하면 해당 규격도 확인합니다.\n\n## 중단 후 다시 실행\n\n같은 데이터·설정·코드와 같은 결과 폴더로 다시 실행하면 `fold_N.ckpt`와 `fold_N_done.json`이 모두 존재하는 **완료된 fold만 재사용**합니다. 실행 도중 끊긴 fold는 처음부터 다시 학습합니다. 마지막 미니배치나 optimizer 상태에서 이어가는 기능은 아닙니다.\n\n데이터·설정·코드의 식별값이 기존 실행과 다르면 혼합된 결과를 만들지 않도록 중단합니다. 이 경우 새 결과 폴더를 지정합니다. 메모리가 부족하면 통화 batch와 window batch를 줄이는 설정을 검토하고 새 폴더에서 시작합니다.\n\n## 대회 조건과 참고 자료\n\n판단 기준은 [docs의 출제 PDF](../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–12쪽입니다. 서울 Training만 학습하고 양측의 전사 `text`만 입력으로 사용합니다. `speaker`, 시간, 주소, 성별, 정답 라벨은 입력 특징에서 제외합니다. 대상은 `고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란`이며, 나머지 라벨만 제거하고 통화는 유지합니다. 외부 추가 데이터와 상용 AI API를 사용하지 않습니다.\n\n최종 제출에는 추론 코드, 학습 로그 포함 노트북, 학습 완료 가중치, 모델 코드, requirements 및 결과 정리 PPT가 필요합니다. 이 구현의 학습을 완료하고 로그·가중치를 보관해야 하며 PPT는 별도로 준비합니다.\n\n공식 구현 참고: [KLUE RoBERTa large](https://huggingface.co/klue/roberta-large), [Transformers tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). 모델 로딩·토큰 처리·혼합 정밀도 API 참고 자료이며 Mission 3의 성능 향상을 보장하는 근거는 아닙니다.\n', 'ver4_runtime/__init__.py': '"""Unmodified ver4 runtime retained for compatible inference and run verification."""\n', 'ver4_runtime/calibration.py': '"""Training-only grouped calibration for nine independent symptom labels."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .data import TARGET_SYMPTOMS\nelse:\n    from data import TARGET_SYMPTOMS\n\nWEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))\n\n\ndef _probabilities(values, name):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):\n        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(f"{name} must be nonempty and finite")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError(f"{name} must contain probabilities in [0, 1]")\n    return values\n\n\ndef _inputs(y_true, neural_probs, sparse_probs):\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    truth = np.asarray(y_true)\n    if truth.shape != neural.shape or sparse.shape != neural.shape:\n        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")\n    if not np.isin(truth, (0, 1)).all():\n        raise ValueError("y_true must contain only binary labels")\n    return truth.astype(np.int8), neural, sparse\n\n\ndef _best_threshold(truth, probabilities):\n    """Exact F1 maximum for >= decisions; tied probabilities stay together."""\n    positives = int(truth.sum())\n    if positives == 0:\n        return ALL_NEGATIVE_THRESHOLD, 0.0\n    if positives == len(truth):\n        return 0.0, 1.0\n    order = np.argsort(-probabilities, kind="stable")\n    sorted_probs = probabilities[order]\n    # A threshold can include a complete tied block, never part of one.\n    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]\n    true_positives = np.cumsum(truth[order])[boundaries]\n    scores = 2.0 * true_positives / (positives + boundaries + 1)\n    thresholds = np.zeros(len(boundaries), dtype=np.float64)\n    has_next = boundaries < len(truth) - 1\n    upper = sorted_probs[boundaries[has_next]]\n    lower = sorted_probs[boundaries[has_next] + 1]\n    midpoint = lower + (upper - lower) / 2\n    # Adjacent floating-point values may have no representable midpoint.\n    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)\n    best = np.flatnonzero(scores == scores.max())\n    # An explicit deterministic tie break preserves the exact maximum.\n    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))\n    return float(thresholds[winner]), float(scores[winner])\n\n\ndef fit_calibrator(y_true, neural_probs, sparse_probs):\n    """Select each label\'s blend and threshold using supplied training labels."""\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    weights, thresholds = [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        candidates = []\n        for weight in WEIGHT_GRID:\n            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]\n            threshold, score = _best_threshold(truth[:, column], blended)\n            candidates.append((score, -abs(weight - 0.5), weight, threshold))\n        _, _, weight, threshold = max(candidates)\n        weights.append(float(weight))\n        thresholds.append(float(threshold))\n    return {"neural_weights": weights, "thresholds": thresholds}\n\n\ndef apply_calibrator(neural_probs, sparse_probs, calibrator):\n    """Return blended probabilities; use >= stored thresholds for labels."""\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    if neural.shape != sparse.shape:\n        raise ValueError("neural_probs and sparse_probs must have identical shapes")\n    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)\n    expected = (len(TARGET_SYMPTOMS),)\n    if weights.shape != expected or thresholds.shape != expected:\n        raise ValueError("calibrator must contain one weight and threshold per symptom")\n    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):\n        raise ValueError("calibrator weights must be finite and in [0, 1]")\n    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):\n        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")\n    return weights * neural + (1.0 - weights) * sparse\n\n\ndef _grouped_splits(truth, groups, seed, n_splits):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) != len(truth):\n        raise ValueError("groups must contain one group ID per sample")\n    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:\n        raise ValueError("n_splits must be an integer >= 2")\n    unique_groups, inverse = np.unique(groups, return_inverse=True)\n    if len(unique_groups) < 2:\n        raise ValueError("cross-fit calibration requires at least two distinct groups")\n    n_splits = min(n_splits, len(unique_groups))\n    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)\n    np.maximum.at(group_targets, inverse, truth)\n    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):\n        train = np.flatnonzero(np.isin(inverse, train_groups))\n        test = np.flatnonzero(np.isin(inverse, test_groups))\n        if not len(train) or not len(test):\n            raise ValueError("grouped calibration produced an empty fold")\n        yield train, test\n\n\ndef calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):\n    """Cross-fit within Training calibration data, then fit deployment settings.\n\n    Base-model predictions must themselves come from samples excluded from\n    base-model training and model selection. Never pass official Validation.\n    """\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    crossfit_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])\n        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)\n        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("each calibration sample must be scored exactly once")\n    final_calibrator = fit_calibrator(truth, neural, sparse)\n    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, crossfit_predictions, average=None, zero_division=0\n    )\n    report = {\n        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",\n        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "calibration_crossfit_macro_f1": float(f1.mean()),\n        "per_symptom_crossfit": {\n            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n    }\n    return final_calibrator, report\n', 'ver4_runtime/data.py': '"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport unicodedata\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""\n    utterances = annotation.get("utterances")\n    if not isinstance(utterances, list):\n        utterances = annotation.get("utterences", [])\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):\n            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n            if text:\n                texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):\n        raise ValueError("symptom must be a string or a list of strings")\n    found = {x.strip() for x in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths:\n        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")\n    if len({p.name for p in paths}) != len(paths):\n        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"JSON object가 아닙니다: {path}")\n        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef grouped_targets(texts, targets):\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse = np.unique(groups, return_inverse=True)\n    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    np.maximum.at(aggregate, inverse, targets)\n    return groups, inverse, aggregate\n\n\ndef make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):\n    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""\n    if not 0 < calibration_size < 0.5 or n_folds < 2:\n        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")\n    groups, inverse, aggregate = grouped_targets(texts, targets)\n    group_ids = np.arange(len(aggregate))\n    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)\n    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))\n    development = np.flatnonzero(np.isin(inverse, dev_groups))\n    calibration = np.flatnonzero(np.isin(inverse, cal_groups))\n    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)\n    folds = []\n    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):\n        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))\n        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))\n        folds.append((train_indices, stop_indices))\n    for name, indices in [("calibration", calibration), ("development", development),\n                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)\n                            for part, ids in zip(("fit", "stop"), pair)]]:\n        counts = targets[indices].sum(axis=0)\n        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):\n            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")\n    return development, calibration, folds, groups\n\n\ndef write_json(path: Path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_predictions(paths, probabilities, thresholds, output: Path):\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid prediction shape or non-finite probabilities")\n    predicted = probabilities >= np.asarray(thresholds)\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predicted):\n            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.\n            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    return predicted.astype(np.int8)\n\n\ndef score_predictions(targets, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i])}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n', 'ver4_runtime/evaluate.py': '"""Score frozen CSV predictions after inference, without changing the model."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .data import write_json\n    from .pipeline import evaluate_csv\nelse:\n    from data import write_json\n    from pipeline import evaluate_csv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--prediction", required=True, type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    report = evaluate_csv(args.label_dir, args.prediction)\n    if args.output:\n        write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/inference.py': '"""Official one-command inference. Mission 3 accepts but does not read audio_dir."""\nimport argparse\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import predict_pipeline\nelse:\n    from pipeline import predict_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", required=True, help="Organizer interface; unused text-only task")\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--ckpt_path", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--device", choices=["cpu", "cuda"])\n    parser.add_argument("--batch_size", type=int)\n    args = parser.parse_args()\n    predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/neural.py': '"""Document-level multi-label transformer; every dialogue window is retained."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\n\nNUM_LABELS = 9\n\n\ndef tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):\n    """Return documents of token windows, including the last window of long calls.\n\n    Only ordinary tokenizer special tokens are used. The stride is an overlap in\n    tokens, not an upper bound on the number of windows or dialogue length.\n    """\n    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if usable_length < 1 or not 0 <= stride < usable_length:\n        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")\n    texts = list(texts)\n    documents = [[] for _ in texts]\n    for start in range(0, len(texts), 128):\n        encoded = tokenizer(\n            texts[start:start + 128], truncation=True, max_length=max_length,\n            stride=stride, return_overflowing_tokens=True, padding=False,\n            return_attention_mask=True,\n        )\n        if "overflow_to_sample_mapping" not in encoded:\n            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")\n        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):\n            documents[start + int(sample)].append({\n                key: encoded[key][window]\n                for key in tokenizer.model_input_names if key in encoded\n            })\n    if any(not windows for windows in documents):\n        raise ValueError("Tokenizer returned a document without any windows")\n    return documents\n\n\nclass DialogueDataset(Dataset):\n    def __init__(self, documents, targets=None):\n        self.documents, self.targets = documents, targets\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["labels"] = self.targets[index]\n        return item\n\n\nclass DialogueCollator:\n    """Flatten windows for the encoder while keeping exactly one target per call."""\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, examples):\n        windows, doc_indices = [], []\n        for index, example in enumerate(examples):\n            if not example["windows"]:\n                raise ValueError("A document must contain at least one token window")\n            windows.extend(example["windows"])\n            doc_indices.extend([index] * len(example["windows"]))\n        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))\n        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)\n        if "labels" in examples[0]:\n            batch["labels"] = torch.from_numpy(np.asarray(\n                [example["labels"] for example in examples], dtype=np.float32,\n            ))\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Label-specific attention pools complete dialogues before computing BCE."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8):\n        super().__init__()\n        self.encoder = encoder\n        self.window_batch_size = int(window_batch_size)\n        if self.window_batch_size < 1:\n            raise ValueError("window_batch_size must be positive")\n        hidden = encoder.config.hidden_size\n        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, doc_indices, **inputs):\n        representations = []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n        windows = self.dropout(torch.cat(representations))\n        logits = []\n        # Each label learns which part of a potentially long call provides evidence.\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected = windows[doc_indices == document_index]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)\n        return torch.stack(logits)\n\n\ndef _device(config):\n    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")\n    return device\n\n\ndef _amp(device):\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):\n    if int(batch_size) < 1:\n        raise ValueError("batch_size must be positive")\n    generator = torch.Generator().manual_seed(seed)\n    return DataLoader(\n        DialogueDataset(documents, targets), batch_size=int(batch_size),\n        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),\n        num_workers=0, generator=generator,\n    )\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _optimizer(model, config):\n    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))\n    decay = float(config.get("layerwise_lr_decay", 0.9))\n    if not 0 < decay <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        if name.startswith("encoder."):\n            match = re.search(r"\\.layer\\.(\\d+)\\.", name)\n            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)\n            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)\n        else:\n            lr = float(config.get("head_learning_rate", 1e-4))\n        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))\n        groups.setdefault((lr, wd), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}\n                             for (lr, wd), params in groups.items()])\n\n\ndef _atomic_save(checkpoint, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _targets(targets, documents, name):\n    targets = np.asarray(targets, dtype=np.float32)\n    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):\n        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")\n    if not np.isin(targets, [0, 1]).all():\n        raise ValueError(f"{name} targets must contain only 0 and 1")\n    return targets\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config: dict, checkpoint_path: Path, seed: int):\n    """Train on documents; select best epoch only by stopping-set macro AP.\n\n    This function does not tune thresholds and never uses calibration targets.\n    GPU enforcement for production belongs to the orchestration entry point.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = _device(config)\n    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))\n    accumulation = int(config.get("gradient_accumulation_steps", 4))\n    if min(max_epochs, patience, accumulation) < 1:\n        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")\n    model_name = config.get("model_name", "klue/roberta-large")\n    encoder = AutoModel.from_pretrained(model_name)\n    if config.get("gradient_checkpointing", True):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),\n                    "window_batch_size": int(config.get("window_batch_size", 8))}\n    model = DialogueClassifier(encoder, **architecture).to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer,\n                           config.get("batch_size", 2), shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, config)\n    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),\n        num_training_steps=total_steps,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    valid_labels = stop_targets.sum(axis=0) > 0\n    if not valid_labels.any():\n        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")\n    if not valid_labels.all():\n        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)\n    best_ap, best_epoch, stale, history = -1.0, 0, 0, []\n    for epoch in range(1, max_epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Match a single full document batch, including a shorter last batch.\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += loss.detach().item() * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            log_every = int(config.get("log_every", 100))\n            if log_every > 0 and (step + 1) % log_every == 0:\n                elapsed = (time.monotonic() - epoch_started) / 60\n                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "\n                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)\n        probabilities = _predict(model, stop_loader, device)\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Stopping predictions contain non-finite values")\n        score = float(average_precision_score(stop_targets[:, valid_labels],\n                                              probabilities[:, valid_labels], average="macro"))\n        row = {"epoch": epoch, "train_loss": loss_sum / document_count,\n               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),\n               "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.5f} minutes={row[\'minutes\']:.1f}", flush=True)\n        if score > best_ap + 1e-6:\n            best_ap, best_epoch, stale = score, epoch, 0\n            _atomic_save({\n                "format_version": 1, "encoder_config": encoder.config.to_dict(),\n                "architecture": architecture, "model_name": str(model_name),\n                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "epoch": epoch, "stop_ap": score, "seed": int(seed),\n            }, checkpoint_path)\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Rebuild from the saved config and weights; no pretrained network fetch."""\n    device = _device(config)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint.get("format_version") != 1:\n        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")\n    encoder_data = dict(checkpoint["encoder_config"])\n    model_type = encoder_data.pop("model_type")\n    encoder_config = AutoConfig.for_model(model_type, **encoder_data)\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    del checkpoint\n    model.to(device)\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))\n    predictions = _predict(model, loader, device)\n    if not np.isfinite(predictions).all():\n        raise FloatingPointError("Inference predictions contain non-finite values")\n    return predictions\n', 'ver4_runtime/pipeline.py': '"""Training-only model selection, ensemble calibration, and portable inference."""\nfrom __future__ import annotations\n\nimport ast\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from .calibration import apply_calibrator, calibrate_crossfit\n    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from .neural import predict_checkpoint, tokenize_dialogues, train_model\nelse:\n    from calibration import apply_calibrator, calibrate_crossfit\n    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from neural import predict_checkpoint, tokenize_dialogues, train_model\n\n\nDEFAULT_CONFIG = {\n    "model_name": "klue/roberta-large",\n    "seed": 42,\n    "n_folds": 5,\n    "calibration_size": 0.10,\n    "calibration_folds": 5,\n    "max_length": 512,\n    "stride": 64,\n    "max_epochs": 6,\n    "patience": 2,\n    "batch_size": 2,\n    "inference_batch_size": 8,\n    "window_batch_size": 2,\n    "gradient_accumulation_steps": 16,\n    "learning_rate": 1.5e-5,\n    "head_learning_rate": 1e-4,\n    "layerwise_lr_decay": 0.95,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.10,\n    "dropout": 0.20,\n    "pos_weight_power": 0.5,\n    "gradient_checkpointing": True,\n    "device": "cuda",\n    "require_cuda": True,\n    "char_features": 120_000,\n    "word_features": 40_000,\n    "min_df": 2,\n    "logistic_c": 4.0,\n}\nRUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")\n\n\ndef fit_sparse(texts, targets, config):\n    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""\n    features = FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                max_features=config["char_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\\b\\w+\\b",\n                                max_features=config["word_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n    matrix = features.fit_transform(texts)\n    classifiers = []\n    for column, name in enumerate(TARGET_SYMPTOMS):\n        truth = targets[:, column]\n        if len(np.unique(truth)) == 1:\n            classifier = float(truth[0])\n        else:\n            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",\n                                            solver="liblinear", max_iter=1500, random_state=config["seed"])\n            classifier.fit(matrix, truth)\n        classifiers.append(classifier)\n        print(f"TF-IDF fit: {name}", flush=True)\n    return {"features": features, "classifiers": classifiers}\n\n\ndef predict_sparse(model, texts):\n    if not texts:\n        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)\n    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.\n    batches = []\n    for start in range(0, len(texts), 1024):\n        matrix = model["features"].transform(texts[start:start + 1024])\n        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)\n                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]\n        batches.append(np.column_stack(columns))\n    return np.concatenate(batches).astype(np.float32)\n\n\ndef _release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _fingerprint(paths, texts, targets, config):\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    for path, text, target in zip(paths, texts, targets):\n        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))\n    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.\n    for name in RUNTIME_FILES:\n        path = Path(__file__).parent / name\n        digest.update(path.name.encode())\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef train_pipeline(train_label_dir, output_dir, config=None):\n    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()\n    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:\n        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")\n    if config["device"] == "cuda":\n        torch.set_float32_matmul_precision("high")\n        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    dev, calibration, folds, groups = make_splits(\n        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    signature = _fingerprint(paths, texts, targets, config)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:\n        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})\n    write_json(output_dir / "splits.json", {\n        "calibration": [paths[i].name for i in calibration],\n        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}\n                  for fit, stop in folds],\n        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",\n    })\n    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)\n    tokenizer_dir = output_dir / "tokenizer"\n    if (tokenizer_dir / "tokenizer_config.json").exists():\n        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)\n        tokenizer.save_pretrained(tokenizer_dir)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    calibration_documents = [documents[i] for i in calibration]\n    counts = np.asarray([len(document) for document in documents])\n    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)\n    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    fold_names, histories = [], []\n    for number, (fit, stop) in enumerate(folds, 1):\n        name = f"fold_{number}.ckpt"\n        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"\n        print(f"\\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)\n        if done_path.exists() and checkpoint_path.exists():\n            result = json.loads(done_path.read_text(encoding="utf-8"))\n            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)\n        else:\n            result = train_model([documents[i] for i in fit], targets[fit],\n                                 [documents[i] for i in stop], targets[stop], tokenizer,\n                                 config, checkpoint_path, seed=config["seed"] + number * 1009)\n            write_json(done_path, result)\n        _release_memory()\n        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)\n        _release_memory()\n        fold_names.append(name)\n        histories.append(result)\n    neural_probs /= len(folds)\n    sparse_path = output_dir / "sparse.joblib"\n    if sparse_path.exists():\n        sparse = joblib.load(sparse_path)\n    else:\n        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)\n        temporary = sparse_path.with_name("sparse.joblib.tmp")\n        joblib.dump(sparse, temporary, compress=3)\n        temporary.replace(sparse_path)\n    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])\n    del sparse\n    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,\n                                          groups[calibration], seed=config["seed"] + 2,\n                                          n_splits=config["calibration_folds"])\n    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.\n    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],\n                        neural=neural_probs, sparse=sparse_probs,\n                        filenames=np.asarray([paths[i].name for i in calibration]))\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),\n                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",\n                "calibrator": calibrator, "signature": signature, "versions": versions}\n    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"\n    _torch_save(checkpoint_path, manifest)\n    report.update({"training_samples": len(paths), "development_samples": len(dev),\n                   "model": f"{config[\'model_name\']} {len(folds)}-fold ensemble + char/word TF-IDF LR",\n                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],\n                   "versions": versions, "validation_used_for_training_or_calibration": False})\n    write_json(output_dir / "mission3_ver4_metrics.json", report)\n    write_json(output_dir / "mission3_ver4_history.json", histories)\n    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):\n        path = Path(__file__).parent / name\n        if path.exists():\n            shutil.copy2(path, submission / path.name)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print("Checkpoint:", checkpoint_path, flush=True)\n    return checkpoint_path, report\n\n\ndef load_manifest(checkpoint_path):\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:\n        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")\n    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:\n        raise ValueError("Checkpoint class order/folds mismatch")\n    root = Path(checkpoint_path).resolve().parent\n    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.exists():\n            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for name in manifest["folds"]:\n        print("Predict:", name, flush=True)\n        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)\n        _release_memory()\n    neural_probs /= len(manifest["folds"])\n    sparse = joblib.load(root / manifest["sparse"])\n    sparse_probs = predict_sparse(sparse, texts)\n    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\ndef evaluate_csv(label_dir, prediction_path):\n    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""\n    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["label file name", "symptom"]:\n            raise ValueError("CSV header must be: label file name,symptom")\n        rows = list(reader)\n    by_name = {}\n    for row in rows:\n        if row["label file name"] in by_name:\n            raise ValueError("Duplicate prediction filename")\n        labels = ast.literal_eval(row["symptom"])\n        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):\n            raise ValueError("CSV symptom must be a list of unique target strings")\n        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]\n    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)\n    if set(by_name) != {path.name for path in paths}:\n        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")\n    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))\n', 'ver4_runtime/requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'ver4_runtime/train.py': '"""Run all Training-only fitting and calibration; never accepts Validation."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--output_dir", required=True, type=Path)\n    parser.add_argument("--config", type=Path, help="Optional JSON overrides of DEFAULT_CONFIG")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding="utf-8")) if args.config else None\n    train_pipeline(args.label_dir, args.output_dir, config)\n\n\nif __name__ == "__main__":\n    main()\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver6_runtime_"))
for name, source in RUNTIME_SOURCES.items():
    path = runtime_dir / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
config_path = runtime_dir / "notebook_config.json"
config_path.write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")

def run_python(arguments):
    command = [sys.executable, "-u", *map(str, arguments)]
    environment = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", env=environment)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except BaseException:
        process.kill()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print("실행 코드:", runtime_dir)
print("새 결과가 저장될 위치:", OUTPUT_DIR)


## 5. 특징 1회 추출 → 분류기 A–F 비교 → 선택 후보 미세조정 → 최종 보정

이 단계는 **Training만 사용**합니다. 기존 split·입력 JSON·모델 가중치·보정 캐시를 먼저 검증합니다.
초기 모델 검사에는 약 6.3GiB의 가중치를 읽는 시간이 들고, token 특징 추출에는 GPU 순전파가 필요합니다.
첫 실행은 여섯 분류기의 학습 시간만으로 전체 소요 시간을 추정할 수 없습니다.
이미 유효한 특징 캐시가 있는 같은 런타임에서는 추출을 생략합니다.

각 분류기는 최대 4 epoch 학습하며 stop Macro F1로 비교합니다. 선택된 분류기는 상위 2개 층을
최대 1 epoch 미세조정하고, 성능이 나빠지면 미세조정 전 가중치를 유지합니다.
**ver6의 최종 `best_epoch=0`은 선택된 분류기를 유지하고 encoder 미세조정만 채택하지 않았다는 뜻입니다.**
분류기 학습이 최종 가중치에 반영됐는지는 A–F 이력의 최적 epoch에서 따로 확인합니다.
분류기는 이미 학습된 epoch가 선택될 수도, 초기 epoch 0이 유지될 수도 있습니다.
최종 확률 평균·logit 평균과 증상별 임계값은 별도 Training calibration에서 CPU로 비교합니다.
아주 작은 차이만 있는 후보가 선택되는 것을 줄이도록 최종 calibration 교차평가 F1이 기존 대비
**0.002 이상 개선**되고 **5개 fold 중 3개 이상에서 개선**될 때만 새 구성을 채택합니다.
이 기준도 공식 Validation 또는 Test 성능의 개선을 보장하지는 않습니다.
정확한 기준과 판단 결과는 `mission3_ver6_metrics.json`에 저장됩니다.

중단 후에는 동일한 설정·데이터·코드로 다시 실행하세요. 완료한 단계/epoch를 재사용하고,
중단된 작업은 해당 단계의 저장 상태에 따라 다시 실행합니다. 새 런타임에서 분류기 비교가 남아 있으면
특징 캐시를 다시 만듭니다. 분류기 비교가 끝났다면 미세조정 재개를 위해 특징을 다시 만들지 않습니다.
**진행 중 학습을 재개할 때는 같은 GPU 종류·AMP 정밀도·관련 런타임 패키지 버전을 사용하세요.**
서로 다른 정밀도로 계산한 stop 점수를 섞지 않도록 실행 조건을 검사합니다.
최종 결과까지 완료된 실행은 다른 GPU에서도 재사용할 수 있으며, 조건이 다른 예측 캐시는 다시 계산합니다.
**다른 설정으로 실험할 때는 첫 셀의 `OUTPUT_DIR`을 새 폴더로 바꾸세요.**


In [ ]:
training_started = time.monotonic()
run_python([
    runtime_dir / "train.py", "--train_label_dir", TRAIN_LABEL_DIR,
    "--base_dir", BASE_DIR, "--output_dir", OUTPUT_DIR,
    "--config_json", config_path, "--auto_batch",
])
TRAINING_MINUTES = (time.monotonic() - training_started) / 60
CHECKPOINT_PATH = OUTPUT_DIR / "mission3_ver6_model.ckpt"
print(f"특징 추출·분류기 비교·미세조정·보정·저장: {TRAINING_MINUTES:.1f}분")
print("ver6 최종 모델 구성 파일:", CHECKPOINT_PATH)


In [ ]:
# 6. Training 내부 선택 결과 — 공식 Validation 점수와 구분합니다.
REPORT_PATH = OUTPUT_DIR / "mission3_ver6_metrics.json"
training_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
candidate_report = training_report.get("candidate_training", {})
trials = candidate_report.get("head_trials", [])
if isinstance(trials, dict):
    trials = list(trials.values())
print("분류기 실험 | 최적 epoch | stop 교차평가 Macro F1 | stop AP")
for trial in trials:
    print(f"{trial.get('profile', '?')} | {trial.get('best_epoch', '?')} | "
          f"{trial.get('best_stop_macro_f1', '기록 확인')} | {trial.get('best_stop_ap', '기록 확인')}")
print("선택된 분류기:", candidate_report.get("selected_profile", "상세 기록 확인"))
print("encoder 미세조정 최적 epoch:", candidate_report.get("best_epoch", "상세 기록 확인"))
if candidate_report.get("best_epoch") == 0:
    print("encoder 미세조정 전의 선택된 분류기 가중치를 유지합니다. 분류기 학습 반영 여부는 위 최적 epoch를 확인하세요.")
print("후보 선택 기준:", candidate_report.get("selection_scope", "원래 fold 1 stop의 threshold 교차평가"))
for field, title in (
    ("baseline_crossfit_macro_f1", "기존 ver4 calibration 교차평가"),
    ("candidate_crossfit_macro_f1", "새 후보 calibration 교차평가"),
    ("candidate_raw_crossfit_macro_f1", "확률 평균 후보"),
    ("candidate_logit_crossfit_macro_f1", "logit 평균 후보"),
    ("selected_name", "최종 선택"),
    ("selected_blend_space", "선택된 혼합 방식"),
    ("selection_gain", "calibration F1 개선 폭"),
    ("minimum_gain", "최소 개선 폭 기준"),
    ("candidate_improved_folds", "F1이 개선된 calibration fold 수"),
    ("minimum_improved_folds", "최소 개선 fold 수 기준"),
    ("candidate_fold_gains", "calibration fold별 개선 폭"),
    ("fallback_reason", "선택 조건 판단"),
):
    if field in training_report:
        print(title + ":", training_report[field])
if training_report.get("selected_name") == "ver4_baseline":
    print("Training 내부의 개선 기준을 충족하지 못해 기존 ver4 보정을 유지합니다. ver6 실행 자체는 완료됐습니다.")
selected = training_report.get("calibrator", {})
labels = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
print("증상 | 기존 신경망 alpha | 새 후보 beta | 임계값")
for label, alpha, beta, threshold in zip(labels, selected.get("neural_weights", []),
                                         selected.get("candidate_weights", []), selected.get("thresholds", [])):
    print(f"{label} | {alpha:.2f} | {beta:.2f} | {threshold:.9f}")
print("실제 계산에는 저장된 보정값 전체 정밀도를 사용합니다.")
print("전체 실험 기록:", REPORT_PATH)


## 7. 확정된 모델로 Validation 전체 예측·평가

여기서 처음으로 공식 Validation 전사 text를 모델에 전달합니다.
먼저 예측 CSV를 확정한 뒤 symptom 정답을 읽어 평가합니다. 이 결과로 모델이나 임계값을 다시 선택하지 않습니다.
기본 서울 Validation은 **3,640개 전체**이며 증상별 precision·recall·F1·support를 함께 표시합니다.
필요하다고 선택된 모델만 추론하고, 입력·가중치·실행 조건이 같은 원시 예측은 공유 캐시에서 재사용합니다.
cache가 없거나 조건이 다르면 첫 실행에서 기존 5개 모델과 새 후보의 추론 비용이 추가될 수 있습니다.

비교 기준: 기존 전체 Validation Macro F1은 **ver4 0.656648948 / ver5 0.656717758**입니다.
두 값은 이전 실행 기록이며 현재 GPU·batch 등에 따른 수치 차이가 있을 수 있습니다.
아래 차이는 참고용 결과 비교이며 학습·보정에 사용하지 않습니다.


In [ ]:
PREDICTION_PATH = None
evaluation_started = time.monotonic()
run_stamp = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
if RUN_VALIDATION and VALID_LABEL_DIR is not None:
    prediction_dir = OUTPUT_DIR / "predictions" / run_stamp
    PREDICTION_PATH = prediction_dir / "mission3.csv"
    run_python([
        runtime_dir / "inference.py", "--audio_dir", "", "--label_dir", VALID_LABEL_DIR,
        "--ckpt_path", CHECKPOINT_PATH, "--output", PREDICTION_PATH,
        *(["--base_dir", BASE_DIR] if not TRAIN_CONFIG.get("bundle_base", True) else []),
        "--batch_size", INFERENCE_BATCH_SIZE, "--cache_dir", INFERENCE_CACHE_DIR, "--evaluate",
    ])
    metrics_path = prediction_dir / "validation_metrics.json"
    if metrics_path.is_file():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"현재 입력 {metrics['samples']:,}개 Macro F1: {metrics['macro_f1']:.9f}")
        for version, reference in (("ver4", 0.6566489480388843), ("ver5", 0.6567177583831921)):
            difference = metrics["macro_f1"] - reference
            print(f"{version} 이전 기록 대비: {difference:+.9f} ({difference * 100:+.5f}%p)")
        print("증상 | Precision | Recall | F1 | 실제 양성 개수")
        for symptom, values in metrics["per_symptom"].items():
            print(f"{symptom} | {values['precision']:.6f} | {values['recall']:.6f} | "
                  f"{values['f1']:.6f} | {values['support']}")
    else:
        print("CSV 생성 완료. 정답이 없는 입력이면 평가 점수는 생성하지 않습니다.")
else:
    print("학습·모델 저장 완료. RUN_VALIDATION=False이므로 최종 평가를 생략했습니다.")
EVALUATION_MINUTES = (time.monotonic() - evaluation_started) / 60
timing = {
    "preparation_minutes": PREPARATION_MINUTES,
    "training_and_calibration_minutes": TRAINING_MINUTES,
    "inference_and_evaluation_minutes": EVALUATION_MINUTES,
    "output_dir": str(OUTPUT_DIR),
    "prediction_csv": str(PREDICTION_PATH) if PREDICTION_PATH else None,
    "note": "벽시계 시간이며 CU 측정값이 아닙니다. 패키지 설치 시간은 제외합니다.",
}
timing_path = OUTPUT_DIR / ("notebook_runtime_" + run_stamp + ".json")
timing_path.write_text(json.dumps(timing, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"최종 추론·평가: {EVALUATION_MINUTES:.1f}분")
print("새 ver6 결과 폴더:", OUTPUT_DIR)
print("ver6 최종 모델 구성:", CHECKPOINT_PATH)
print("예측 CSV:", PREDICTION_PATH)
print("실행 시간 기록:", timing_path)
print("추론 실행 코드:", OUTPUT_DIR / "submission/inference.py")


## 결과 위치와 다음 실행

**새 결과: `내 드라이브/DCC_AIdeas/results/mission3/ver6`**

- `mission3_ver6_model.ckpt`: 최종 선택된 모델 구성·혼합 비율·임계값.
- `candidate_best.ckpt`: 선택된 분류기와 encoder 가중치.
- `mission3_ver6_metrics.json`: A–F 실험, 미세조정, 내부 교차평가와 최종 선택 기록.
- `predictions/실행시각/mission3.csv`: 제출 형식 예측. 정답이 있으면 같은 폴더에 `validation_metrics.json`.
- `notebook_runtime_실행시각.json`: 입력 준비·학습·최종 추론 단계의 실제 시간. CU는 Colab에서 별도 확인합니다.
- `submission/`: 추론 코드, 패키지 목록, README와 논문 검토 문서.

`/content/mission3_ver5_inputs/.../ver4`는 기존 입력 모델이고 `/content/mission3_ver6_features`는
재계산 가능한 특징 캐시입니다. 새 학습 결과 폴더와 혼동하지 마세요.
기존 5개 모델을 Drive에 중복 저장하지 않으므로 **입력 ZIP과 ver6 결과를 모두 보관해야 합니다.**
같은 런타임에서는 검증된 압축 해제 파일·특징을 재사용합니다.
새 런타임에서는 입력 ZIP을 다시 풉니다. 미완료 분류기 실험이 있을 때만 특징을 다시 계산하며,
이미 저장된 분류기·미세조정 상태와 완료 결과는 검증 후 재사용합니다.
진행 중 학습 재개에는 같은 GPU 종류·AMP 정밀도·런타임 버전이 필요합니다. 완료된 최종 결과는 다른 GPU에서도 사용할 수 있습니다.

다른 환경에서 추론할 때는 ZIP을 푼 ver4 폴더를 함께 지정하세요.

```bash
python submission/inference.py --audio_dir "WAV 폴더" --label_dir "JSON 폴더" \
  --ckpt_path mission3_ver6_model.ckpt --base_dir "복원한 ver4 폴더" --output mission3.csv
```

`--audio_dir`는 대회 인터페이스 호환용이고 WAV는 읽지 않습니다.
최종 내부 선택이 `ver4_baseline`이면 새 실험의 개선 기준을 충족하지 못해 기존 구성을 유지했다는 뜻입니다.
실행 로그가 남은 이 노트북도 보관하세요. 실제 F1 개선 여부는 마지막 전체 Validation 결과로 확인합니다.
